# Lab 3-2 · เปรียบเทียบเทคนิคอิงเนื้อหาบนข้อมูลจริง MovieLens (ml-latest-small) ด้วยการทดลองแบบออฟไลน์

ต่อจาก Lab 3-1 (MovieMini-12 ซึ่งเล็กเกินกว่าจะสรุปสถิติได้) สมุดบันทึกนี้ใช้ **Open Dataset จริง**:
**MovieLens ml-latest-small** (GroupLens · ผู้ใช้ 610 คน · ภาพยนตร์ 9,742 เรื่อง · คะแนน 100,836 รายการ 0.5–5 ดาว
พร้อม timestamp · แนว · และ tag ที่ผู้ใช้ติด 3,683 รายการ) เพื่อ

1. แสดง **เทคนิคการเลือกกลุ่มตัวอย่างสำหรับ Offline Training/Testing** (บทที่ 2): k-core · การแบ่งข้อมูล 4 แบบ ·
   ตรวจการรั่วไหล · Train/Validation/Test · Candidate set (full vs sampled) · negative sampling สำหรับฝึก · seed
2. สร้างและ **เปรียบเทียบเทคนิคของบทที่ 3** กว่า 12 แบบ (TF-IDF · Rocchio · BM25 · LSA · Item-kNN · NB · LR · Hybrid · MMR)
3. **ประเมินผลตามบทที่ 2 ทั้ง Rating (MAE/RMSE) และ Ranking** (P/R/HR/MRR/MAP/nDCG/AUC) และ Beyond-Accuracy
   (Coverage/Gini/LongTail/ILD/Novelty/Calibration/Fairness) พร้อม **ตารางและกราฟ** และ **ตัวอย่างผลลัพธ์บนชุดตรวจสอบ**
4. **ฟอร์ม GUI** สาธิตการประยุกต์ใช้ (ข้อมูลผู้ใช้ + รายการที่แนะนำพร้อมเนื้อหา)

| Part | เนื้อหา |
|---|---|
| 0 | ติดตั้ง · แกนกลาง `cbrec_core` (ฝัง) · ค่าตั้งค่าการทดลอง |
| 1 | โหลด MovieLens · สร้าง "เนื้อหา" ของรายการ (ชื่อ + แนว + tag) · ตรวจสอบข้อมูล (audit) |
| 2 | การแทนเนื้อหา: TF-IDF · เมทริกซ์แนว · BM25 · LSA (· Sentence-Embedding ทางเลือก) |
| 3 | **การเลือกกลุ่มตัวอย่างและการแบ่งข้อมูล**: k-core · 4 โปรโตคอล · leakage · train/valid/test · Protocol Card |
| 4 | โมเดลทั้งหมด (score function ต่อผู้ใช้) และ negative sampling สำหรับฝึกตัวจำแนก |
| 5 | ประเมินแบบ **Ranking** บน Validation: ตาราง · กราฟ · full vs sampled candidates · seed · bootstrap · Holm · fairness |
| 6 | ประเมินแบบ **Rating** (MAE/RMSE) บน Validation + ตัวอย่างค่าทำนายรายคู่ |
| 7 | **ตัวอย่างผลลัพธ์** บน Validation: ประวัติ · ของจริง · Top-K ของทุกโมเดล เทียบกัน (✔) |
| 8 | ทดสอบครั้งเดียวบน Test (ฝึกใหม่ด้วย train+valid) + CI |
| 9 | **ฟอร์ม GUI** |
| 10 | สรุปข้อสังเกต |

> **Colab**: รันทุกเซลล์ตามลำดับ · ดาวน์โหลดข้อมูลอัตโนมัติ (~1 MB) · เวลารันทั้งหมดราว 3–6 นาที (CPU)
> ตั้ง `FAST_MODE = True` ใน Part 0 เพื่อสุ่มผู้ใช้ 200 คน (รันใน ~1 นาที) เมื่อต้องการทดลองเร็ว ๆ

## Part 0 · ติดตั้ง นำเข้า และค่าตั้งค่าการทดลอง

In [ ]:
import importlib, subprocess, sys, os, io, zipfile, urllib.request
for pkg in ["ipywidgets"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180); pd.set_option("display.max_colwidth", 70)

# ---------- ค่าตั้งค่าการทดลอง (ประกาศก่อนดูผล — นิยาม 2.9 Protocol Card) ----------
FAST_MODE   = False      # True = สุ่มผู้ใช้ 200 คนเพื่อรันเร็ว (ผลไม่ใช่ตัวแทนของทั้งชุด)
SEED        = 2569
K           = 10         # ค่า K ของทุกตัวชี้วัด @K (หน้าเว็บแสดง 10 รายการ)
LIKE_THR    = 4.0        # ความเกี่ยวข้อง = rating ≥ 4 (ประกาศ)
DISLIKE_THR = 2.0        # ตัวอย่างลบชัดเจน = rating ≤ 2 (ใช้กับ Rocchio/NB/LR)
K_CORE      = 5          # กรองผู้ใช้/รายการที่มีปฏิสัมพันธ์ < 5 ออก "ก่อน" แบ่งข้อมูล
USE_SBERT   = False      # True = ลองใช้ sentence-transformers (ต้องมีเน็ต/ดาวน์โหลดโมเดล ~90 MB) มิฉะนั้นใช้ LSA แทน
DATA_DIR    = "data/ml-latest-small"

In [ ]:
# ===================== cbrec_core (ฝัง) — เริ่ม =====================
# -*- coding: utf-8 -*-
"""
==========================================================================================
 cbrec_core.py — แกนกลางของระบบผู้แนะนำอิงเนื้อหา (Content-Based) สำหรับบทที่ 3
                 + ตัวชี้วัดการประเมินผล (Accuracy / Beyond-Accuracy) ของบทที่ 2
                 + เครื่องมือออกแบบการทดลองแบบออฟไลน์ (Offline Experiment)
==========================================================================================
ไฟล์นี้ถูก "ฝัง" อยู่ในสมุดบันทึกทุกเล่ม (Lab3_1 / Lab3_2 / Lab3_3) เพื่อให้แต่ละเล่มรันบน
Google Colab ได้โดยอัปโหลดไฟล์เดียว และยังบันทึกแยกเป็น cbrec_core.py เพื่อ import ซ้ำได้

โครงสร้าง (เรียงตามลำดับของบทที่ 3 → บทที่ 2)
  ส่วนที่ 1  การเตรียมข้อความ (3.4–3.5)   : tokenize · คำหยุด · n-gram
  ส่วนที่ 2  การถ่วงน้ำหนักคำ (3.6–3.7)   : TF 5 รูปแบบ · IDF หลายสูตร · L2 · โคไซน์
  ส่วนที่ 3  BM25 / BM25+ (3.11)          : การอิ่มตัวและปรับความยาวเอกสาร
  ส่วนที่ 4  ปริภูมิความหมาย (3.13)         : LSA (TruncatedSVD) · Sentence-Embedding (ทางเลือก)
  ส่วนที่ 5  โปรไฟล์ผู้ใช้ (3.8, 3.12)      : centroid · ถ่วงคะแนน · Rocchio · Naive Bayes · Logistic
  ส่วนที่ 6  ฟังก์ชันจับคู่และจัดอันดับ (3.9–3.10) : โคไซน์ · Item-kNN · Top-K · MMR
  ส่วนที่ 7  ผสานคะแนนและปรับเทียบ (3.18)  : convex · RRF · Platt · ECE
  ส่วนที่ 8  โมเดลทำนายคะแนน (Rating)      : ค่าเฉลี่ย · Content-kNN regression
  ส่วนที่ 9  ตัวชี้วัดบทที่ 2               : MAE/RMSE · P/R/F1/HR@K · MRR · MAP · nDCG · AUC
                                            Coverage · Gini · LongTail · ILD · Novelty · Calibration · Fairness
  ส่วนที่ 10 การออกแบบการทดลองออฟไลน์ (2.7–2.10, 2.14) : split 4 แบบ · leakage check · k-core ·
                                            candidate set · seed variance · paired bootstrap · Holm · ProtocolCard
  ส่วนที่ 11 การนำเสนอผลและ GUI           : ตารางเทียบโมเดล · กราฟ · รายงานผู้ใช้ · ฟอร์ม ipywidgets

หมายเหตุด้านการทำซ้ำ (Reproducibility)
  * ทุกฟังก์ชันที่มีการสุ่มรับพารามิเตอร์ seed และใช้ numpy.random.default_rng(seed)
  * สูตรตั้งต้นทุกตัวตรงกับที่ใช้คำนวณตัวเลขในบทที่ 3 (TF = 1+log2 f, IDF = ln((1+N)/(1+df))+1, L2)
    หากเปลี่ยนสูตร ตัวเลขจะเปลี่ยนทันที — ต้อง "ประกาศการตั้งค่า" ทุกครั้งที่รายงานผล (ตาราง 3.3)
"""
import math
import re
import time
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings("ignore")

SEED = 2569  # ค่าตั้งต้นของ seed ทั้งชุด (พ.ศ. 2569) — เปลี่ยนแล้วผลเปลี่ยน แต่ข้อสรุปเชิงแนวคิดควรคงเดิม


def set_seed(seed=SEED):
    """ตั้ง seed ของ numpy และ random (และ torch หากมี) ให้รันซ้ำได้ผลเดิม"""
    import random
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except Exception:
        pass
    return np.random.default_rng(seed)


class Timer:
    """ตัวจับเวลาอย่างง่าย: with Timer('ชื่อ'): ...  → พิมพ์เวลาที่ใช้ (ใช้ดูต้นทุนของแต่ละขั้น)"""
    def __init__(self, label=""):
        self.label = label
    def __enter__(self):
        self.t0 = time.time(); return self
    def __exit__(self, *a):
        self.sec = time.time() - self.t0
        print(f"[เวลา] {self.label}: {self.sec:.2f} วินาที")


# =========================================================================================
# ส่วนที่ 1  การเตรียมข้อความ (หัวข้อ 3.4–3.5)
# =========================================================================================
# รายการคำหยุดชุดเดียวกับที่ใช้คำนวณตัวเลขทุกตัวในบทที่ 3 (ตาราง 3.3: 457 → 263 โทเคน, 249 → 203 คำ)
# ข้อสังเกต: รายการนี้ "ตั้งใจ" เก็บคำปฏิเสธบางคำ (เช่น not, no) ไว้ในรายการคำหยุดตามต้นฉบับของบท
#           หากนำไปใช้กับงานวิเคราะห์ความรู้สึก ต้องนำ not/no/never ออกจากรายการ (หัวข้อ 3.4.6)
STOP_EN = set("""
a an the and or of to in on at for with from as is are was were be been being am
his her their its it he she they them him us we our you your i my me
who which that this these those what when where why how whom whose
not no nor but by into across over under out up down off above below through
while after before during until since again once here there
must can could may might will would shall should do does did have has had
so such than then too very just only own each every all both few more most other same
one two another between within
""".split())


def tokenize(text, keep_thai=False):
    """ตัดคำแบบง่ายที่สุด: แปลงเป็นตัวพิมพ์เล็กแล้วเก็บเฉพาะสายอักษรละติน [a-z]+
    (ตรงกับกฎที่ใช้ในบทที่ 3 ทำให้ "world." → world, "job:" → job, "Mad Max: Fury Road" → mad max fury road)
    keep_thai=True จะเก็บกลุ่มอักขระไทยไว้ด้วย (ยังไม่ตัดคำไทย — ต้องใช้ pythainlp ในหัวข้อ 3.5)
    """
    text = re.sub(r"'s\b", "", text.lower())     # ตัด 's แสดงความเป็นเจ้าของ (potter's → potter) กันโทเคน "s" ลอย
    if keep_thai:
        return re.findall(r"[a-z]+|[฀-๿]+", text)
    return re.findall(r"[a-z]+", text)


def preprocess(text, stop=STOP_EN, ngram_range=(1, 1), keep_thai=False, stemmer=None):
    """กระบวนงานเตรียมข้อความ (ภาพที่ 3.5): ตัดคำ → ตัดคำหยุด → (ลดรูปคำ) → (เพิ่ม n-gram)
    คืนรายการโทเคน (หน่วยที่จะกลายเป็น "มิติ" ของเวกเตอร์)
    * stemmer: ฟังก์ชัน str→str (เช่น nltk PorterStemmer().stem) ค่าตั้งต้น None = ไม่ลดรูปคำ
    * ngram_range=(1,2) จะเพิ่ม bigram ในรูป "w1_w2" ต่อท้าย unigram (ตาราง 3.11)
    """
    toks = [t for t in tokenize(text, keep_thai) if t not in stop]
    if stemmer is not None:
        toks = [stemmer(t) for t in toks]
    lo, hi = ngram_range
    out = []
    for n in range(lo, hi + 1):
        if n == 1:
            out.extend(toks)
        else:
            out.extend("_".join(toks[i:i + n]) for i in range(len(toks) - n + 1))
    return out


def corpus_stats(token_docs):
    """สถิติของคลัง (ตาราง 3.3): จำนวนเอกสาร โทเคนรวม ขนาดคลังศัพท์ ความยาวเฉลี่ย avgdl"""
    n_tok = sum(len(d) for d in token_docs)
    vocab = set(t for d in token_docs for t in d)
    return {"N (เอกสาร)": len(token_docs), "โทเคนรวม": n_tok, "ขนาดคลังศัพท์": len(vocab),
            "avgdl": n_tok / max(1, len(token_docs))}


# =========================================================================================
# ส่วนที่ 2  การถ่วงน้ำหนักคำ TF / IDF / L2 และโคไซน์ (หัวข้อ 3.6–3.7)
# =========================================================================================
def tf_weight(f, scheme="log", max_f=1, doc_len=1):
    """TF ห้ารูปแบบ (ตาราง 3.20)
    raw    : f                      นับตรง ๆ (คำที่ซ้ำ 3 ครั้งได้ 3 เท่า — ไม่มีเพดาน)
    log    : 1 + log2(f)            ลดทอนแบบลอการิทึม (ค่าตั้งต้นของบท; sublinear_tf)
    binary : 1 ถ้า f>0              สนใจแค่ "มี/ไม่มี"
    aug    : 0.5 + 0.5 f/max_f      ปรับตามคำที่ถี่ที่สุดในเอกสาร (กันเอกสารยาวได้เปรียบ)
    norm   : f / doc_len            สัดส่วนต่อความยาวเอกสาร
    """
    if f <= 0:
        return 0.0
    if scheme == "raw":
        return float(f)
    if scheme == "log":
        return 1.0 + math.log2(f)
    if scheme == "binary":
        return 1.0
    if scheme == "aug":
        return 0.5 + 0.5 * f / max(1, max_f)
    if scheme == "norm":
        return f / max(1, doc_len)
    raise ValueError(scheme)


def idf_weight(df, N, scheme="smooth"):
    """IDF หลายสูตร (ตาราง 3.24 — TF-IDF เป็น "ตระกูล" ของสูตร ไม่ใช่สูตรเดียว)
    smooth   : ln((1+N)/(1+df)) + 1   ค่าตั้งต้นของบท (เหมือน sklearn smooth_idf) — ไม่มีคำใดได้ 0
    standard : ln(N/df)               สูตรดั้งเดิม คำที่พบทุกเอกสารได้ 0 พอดี
    prob     : max(0, ln((N-df)/df))  แบบความน่าจะเป็น
    bm25     : ln((N-df+0.5)/(df+0.5) + 1)  สูตรที่ BM25 (Lucene) ใช้
    none     : 1                      ไม่ถ่วง IDF (เท่ากับ TF อย่างเดียว)
    """
    if scheme == "smooth":
        return math.log((1 + N) / (1 + df)) + 1.0
    if scheme == "standard":
        return math.log(N / df)
    if scheme == "prob":
        return max(0.0, math.log((N - df) / df)) if df < N else 0.0
    if scheme == "bm25":
        return math.log((N - df + 0.5) / (df + 0.5) + 1.0)
    if scheme == "none":
        return 1.0
    raise ValueError(scheme)


class TextVectorizer:
    """เวกเตอร์ไรเซอร์ที่เขียนเองเพื่อให้ "เห็นทุกขั้น" (ต่างจาก sklearn ตรงที่เลือกสูตร TF/IDF ได้อิสระ)

    พารามิเตอร์ (ตาราง 3.67)
      tf, idf        : รูปแบบ TF และ IDF (ดู tf_weight / idf_weight)
      min_df, max_df : ตัดคำที่พบน้อยเกิน (จำนวนเอกสาร) / บ่อยเกิน (สัดส่วนเอกสาร) — คุมขนาดคลังศัพท์
      ngram_range    : (1,1) unigram · (1,2) unigram+bigram
      norm           : 'l2' (ค่าตั้งต้น — ทำให้ผลคูณจุด = โคไซน์) หรือ None
      max_features   : เก็บเฉพาะคำที่ df สูงสุด N คำ (None = ทั้งหมด)
    หลัง fit จะมี  vocab_ (list), idx_ (dict คำ→คอลัมน์), df_ (Counter), idf_ (np.array),
                   doc_len_ (ความยาวเอกสารฝึกหลังเตรียมข้อความ), avgdl_
    ข้อสังเกต: IDF และ df คำนวณจาก "เอกสารที่ fit" เท่านั้น — ในการทดลองต้อง fit ด้วยคลังรายการ
               (ซึ่งไม่ใช่ข้อมูลผู้ใช้จึงไม่ถือว่ารั่ว) แต่สถิติของผู้ใช้ทุกตัวต้องมาจากชุดฝึก (ตาราง 2.10)
    """
    def __init__(self, tf="log", idf="smooth", min_df=1, max_df=1.0, ngram_range=(1, 1),
                 stop=STOP_EN, norm="l2", max_features=None, keep_thai=False, stemmer=None):
        self.tf, self.idf, self.min_df, self.max_df = tf, idf, min_df, max_df
        self.ngram_range, self.stop, self.norm = ngram_range, stop, norm
        self.max_features, self.keep_thai, self.stemmer = max_features, keep_thai, stemmer

    def _tokens(self, docs):
        return [preprocess(d, self.stop, self.ngram_range, self.keep_thai, self.stemmer) for d in docs]

    def fit(self, docs):
        toks = self._tokens(docs)
        N = len(toks)
        df = Counter()
        for d in toks:
            df.update(set(d))
        max_df_abs = self.max_df * N if isinstance(self.max_df, float) else self.max_df
        vocab = [t for t, c in df.items() if c >= self.min_df and c <= max_df_abs]
        if self.max_features:
            vocab = sorted(vocab, key=lambda t: (-df[t], t))[: self.max_features]
        self.vocab_ = sorted(vocab)                 # เรียงตามตัวอักษรเพื่อให้ตรวจสอบด้วยมือได้ (ตาราง 3.13)
        self.idx_ = {t: k for k, t in enumerate(self.vocab_)}
        self.df_ = df
        self.N_ = N
        self.idf_ = np.array([idf_weight(df[t], N, self.idf) for t in self.vocab_])
        self.doc_len_ = np.array([len(d) for d in toks])
        self.avgdl_ = float(self.doc_len_.mean()) if N else 0.0
        self._train_tokens = toks
        return self

    def transform(self, docs, tokens=None):
        """คืน scipy.sparse CSR ขนาด (จำนวนเอกสาร × ขนาดคลังศัพท์)"""
        toks = tokens if tokens is not None else self._tokens(docs)
        rows, cols, vals = [], [], []
        for r, d in enumerate(toks):
            c = Counter(t for t in d if t in self.idx_)
            if not c:
                continue
            max_f = max(c.values())
            for t, f in c.items():
                j = self.idx_[t]
                rows.append(r); cols.append(j)
                vals.append(tf_weight(f, self.tf, max_f, len(d)) * self.idf_[j])
        X = sp.csr_matrix((vals, (rows, cols)), shape=(len(toks), len(self.vocab_)), dtype=np.float64)
        if self.norm == "l2":
            X = l2_normalize_rows(X)
        return X

    def fit_transform(self, docs):
        self.fit(docs)
        return self.transform(docs, tokens=self._train_tokens)

    def top_terms(self, vec, k=10):
        """คำที่มีน้ำหนักสูงสุด k คำของเวกเตอร์หนึ่งแถว (ใช้ "อ่าน" โปรไฟล์หรือเอกสาร — หัวข้อ 3.9.4)"""
        v = np.asarray(vec.todense()).ravel() if sp.issparse(vec) else np.asarray(vec).ravel()
        order = np.argsort(-v)[:k]
        return [(self.vocab_[j], round(float(v[j]), 4)) for j in order if v[j] > 0]


def l2_normalize_rows(X):
    """ปรับความยาวแถวเป็น 1 (L2) — เอกลักษณ์ 3.7.2: หลัง L2 แล้ว ผลคูณจุด = โคไซน์ และ ‖a−b‖² = 2(1−cos)
    แถวที่เป็นศูนย์ทั้งแถว (เอกสารไม่มีคำในคลังศัพท์) จะคงเป็นศูนย์ ไม่หารด้วยศูนย์ (ตาราง 3.68 — NaN)"""
    if sp.issparse(X):
        X = sp.csr_matrix(X, dtype=np.float64)
        n = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())
        n[n == 0] = 1.0
        return sp.diags(1.0 / n) @ X
    X = np.asarray(X, dtype=np.float64)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return X / n


def cosine_matrix(X, Y=None):
    """เมทริกซ์โคไซน์ทุกคู่ (X: n×d, Y: m×d) — คืน dense n×m ; ถ้า X ผ่าน L2 แล้ว = X·Yᵀ"""
    Xn = l2_normalize_rows(X)
    Yn = Xn if Y is None else l2_normalize_rows(Y)
    S = Xn @ Yn.T
    return np.asarray(S.todense()) if sp.issparse(S) else np.asarray(S)


def to_dense_row(v):
    return np.asarray(v.todense()).ravel() if sp.issparse(v) else np.asarray(v, dtype=np.float64).ravel()


# =========================================================================================
# ส่วนที่ 3  BM25 / BM25+ (หัวข้อ 3.11)
# =========================================================================================
class BM25:
    """BM25 (Robertson) พร้อมส่วนขยาย BM25+ (delta) และเลือก IDF ได้
        score(q, d) = Σ_{t∈q} idf(t) · [ f(t,d)(k1+1) / (f(t,d) + k1(1 − b + b·|d|/avgdl)) + delta ]
    k1 คุมความอิ่มตัวของความถี่ (0.5–3.0 ค่าตั้งต้น 1.5), b คุมการปรับความยาว (0–1 ค่าตั้งต้น 0.75)
    delta>0 คือ BM25+ (ป้องกันเอกสารยาวถูกลดคะแนนมากเกินควร — หัวข้อ 3.11.4)
    ตัวเลขในตาราง 3.46 ใช้ k1=1.5, b=0.75, idf แบบ bm25: ln((N−df+0.5)/(df+0.5)+1)
    """
    def __init__(self, k1=1.5, b=0.75, delta=0.0, idf="bm25"):
        self.k1, self.b, self.delta, self.idf_scheme = k1, b, delta, idf

    def fit(self, token_docs):
        self.docs = token_docs
        self.N = len(token_docs)
        self.doc_len = np.array([len(d) for d in token_docs], dtype=np.float64)
        self.avgdl = self.doc_len.mean() if self.N else 0.0
        self.df = Counter()
        for d in token_docs:
            self.df.update(set(d))
        self.tf = [Counter(d) for d in token_docs]
        self.idf = {t: idf_weight(c, self.N, self.idf_scheme) for t, c in self.df.items()}
        # เมทริกซ์ความถี่คำแบบเบาบาง (N × V) เก็บเป็น CSC เพื่อดึง "คอลัมน์ของคำ" ได้เร็ว — ทำให้ให้คะแนนข้อคำถามยาว ๆ ได้ทันที
        self.vocab_ = sorted(self.df)
        self.idx_ = {t: j for j, t in enumerate(self.vocab_)}
        rows, cols, vals = [], [], []
        for r, c in enumerate(self.tf):
            for t, f in c.items():
                rows.append(r); cols.append(self.idx_[t]); vals.append(f)
        self.TF_ = sp.csc_matrix((vals, (rows, cols)), shape=(self.N, len(self.vocab_)), dtype=np.float64)
        self.K_ = self.k1 * (1 - self.b + self.b * self.doc_len / max(self.avgdl, 1e-9))   # ตัวหารส่วนปรับความยาว ต่อเอกสาร
        return self

    def score_query(self, query_terms, term_weights=None):
        """คะแนน BM25 ของ "ทุกเอกสาร" ต่อข้อคำถามหนึ่งชุด (query_terms = รายการคำ; น้ำหนักคำใช้ได้)
        คืน np.array ยาว N — เอกสารที่ไม่มีคำในข้อคำถามเลยจะได้ 0 พอดี (ต่างจากโคไซน์ที่ได้ค่าบวกเล็ก ๆ จากคำอื่น)"""
        s = np.zeros(self.N)
        for t in query_terms:
            j = self.idx_.get(t)
            if j is None:
                continue
            w = 1.0 if term_weights is None else term_weights.get(t, 1.0)
            col = self.TF_[:, j]                                   # เฉพาะเอกสารที่มีคำ t (เบาบาง)
            docs, f = col.indices, col.data
            comp = f * (self.k1 + 1) / (f + self.K_[docs]) + self.delta
            s[docs] += w * self.idf[t] * comp
        return s

    def explain(self, query_terms, doc_index):
        """แจกแจงองค์ประกอบรายคำของเอกสารหนึ่งฉบับ (ใช้อ่านตาราง 3.46 ทีละคอลัมน์)"""
        rows = []
        K = self.k1 * (1 - self.b + self.b * self.doc_len[doc_index] / max(self.avgdl, 1e-9))
        for t in query_terms:
            f = self.tf[doc_index].get(t, 0)
            if f == 0:
                continue
            comp = f * (self.k1 + 1) / (f + K) + self.delta
            rows.append({"คำ": t, "f": f, "องค์ประกอบ TF": round(comp, 4), "idf": round(self.idf[t], 4),
                         "คะแนน": round(comp * self.idf[t], 4)})
        return pd.DataFrame(rows)


# =========================================================================================
# ส่วนที่ 4  ปริภูมิความหมาย: LSA และ Sentence Embedding (หัวข้อ 3.13)
# =========================================================================================
class LSA:
    """Latent Semantic Analysis = TruncatedSVD บนเมทริกซ์ TF-IDF แล้วปรับ L2
    k (มิติแฝง) ต้องน้อยกว่า min(จำนวนเอกสาร, ขนาดคลังศัพท์) — ในคลังเล็กอย่าง MovieMini-12 (N=12)
    ค่า k จึงเล็กมากและ "ไม่มีสถิติพอ" ให้เชื่อ (ข้อจำกัดที่บทตั้งใจให้เห็น หัวข้อ 3.13.2)"""
    def __init__(self, k=50, seed=SEED):
        self.k, self.seed = k, seed

    def fit(self, X):
        from sklearn.decomposition import TruncatedSVD
        k = int(min(self.k, min(X.shape) - 1))
        self.svd = TruncatedSVD(n_components=max(1, k), random_state=self.seed).fit(X)
        self.explained_ = self.svd.explained_variance_ratio_.sum()
        return self

    def transform(self, X):
        return l2_normalize_rows(self.svd.transform(X))

    def fit_transform(self, X):
        return self.fit(X).transform(X)


def sentence_embeddings(texts, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=64):
    """เวกเตอร์ความหมายจาก Bi-Encoder สำเร็จรูป (ต้อง pip install sentence-transformers และมีอินเทอร์เน็ต)
    คืน np.array (n × d) ที่ปรับ L2 แล้ว ; หากโหลดไม่ได้จะคืน None ให้ผู้เรียกใช้ LSA แทน
    ข้อสังเกต: ผลลัพธ์ขึ้นกับรุ่นโมเดลและ "กลยุทธ์การรวมค่า" (mean pooling) ที่โมเดลนั้นใช้ (3.13.5)"""
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(model_name)
        E = m.encode(list(texts), batch_size=batch_size, show_progress_bar=False, normalize_embeddings=True)
        return np.asarray(E, dtype=np.float64)
    except Exception as e:  # ไม่มีไลบรารี / ไม่มีเน็ต
        print("[คำเตือน] ใช้ sentence-transformers ไม่ได้:", str(e)[:120], "→ ใช้ LSA แทน")
        return None


# =========================================================================================
# ส่วนที่ 5  โปรไฟล์ผู้ใช้ (หัวข้อ 3.8 และ 3.12)
# =========================================================================================
def build_profile(X, hist, method="centroid", like_thr=4, dislike_thr=2, beta=0.8, gamma=0.2,
                  alpha=0.0, query_vec=None, clip_negative=True, normalize=True):
    """สร้างเวกเตอร์โปรไฟล์ในปริภูมิเดียวกับรายการ (หลักการข้อเดียวของ 3.8.1)

    X      : เมทริกซ์รายการ (n_items × d) — sparse หรือ dense ; แถวควรผ่าน L2 แล้ว
    hist   : dict {item_index: rating} ประวัติของผู้ใช้ (ดัชนีแถวของ X)
    method :
      'centroid' : ค่าเฉลี่ยของเวกเตอร์รายการที่ชอบ (rating ≥ like_thr)                     (3.8.2)
      'weighted' : Σ (r−3)·v / Σ|r−3|  ใช้ทั้งชอบและไม่ชอบ ค่ากลาง (3) ไม่มีผล               (3.8.4)
      'rocchio'  : α·q + β·mean(ชอบ) − γ·mean(ไม่ชอบ)  ค่าตั้งต้น β=0.8, γ=0.2 (กฎ β>γ)       (3.8.5)
    clip_negative=True : ตัดมิติที่ติดลบเป็น 0 ก่อนปรับ L2 (ตามตัวอย่างในหัวข้อ 3.8.6)
    คืน np.array ยาว d (ปรับ L2 แล้วเมื่อ normalize=True) — ถ้าไม่มีรายการที่ชอบเลยคืนเวกเตอร์ศูนย์
    ข้อสังเกต: เวกเตอร์ศูนย์ = "ไม่มีหลักฐาน" ต้องแยกจากคะแนนต่ำ (3.10.2) ผู้เรียกควร fallback เป็นความนิยม
    """
    d = X.shape[1]
    liked = [i for i, r in hist.items() if r >= like_thr]
    disliked = [i for i, r in hist.items() if r <= dislike_thr]

    def mean_rows(idx):
        if not idx:
            return np.zeros(d)
        M = X[idx]
        return to_dense_row(M.mean(axis=0))

    if method == "centroid":
        p = mean_rows(liked)
    elif method == "weighted":
        num = np.zeros(d); den = 0.0
        for i, r in hist.items():
            w = r - 3.0
            if w == 0:
                continue
            num += w * to_dense_row(X[i]); den += abs(w)
        p = num / den if den > 0 else num
    elif method == "rocchio":
        q = np.zeros(d) if query_vec is None else to_dense_row(query_vec)
        p = alpha * q + beta * mean_rows(liked) - gamma * mean_rows(disliked)
    else:
        raise ValueError(method)
    if clip_negative:
        p = np.clip(p, 0, None)
    if normalize:
        n = np.linalg.norm(p)
        if n > 0:
            p = p / n
    return p


class ClassifierProfile:
    """โปรไฟล์แบบ "เรียนรู้ได้" รายผู้ใช้ (หัวข้อ 3.12): Multinomial Naive Bayes หรือ Logistic Regression
    มองการแนะนำเป็นการจำแนกสองคลาส: ชอบ (rating ≥ like_thr) กับ ไม่ชอบ (≤ dislike_thr)
      * NB ต้องใช้ "จำนวนนับ" (เมทริกซ์ที่ค่าไม่ติดลบ) และมีการปรับเรียบลาปลัส alpha (นิยาม 3.11)
      * LR เรียนน้ำหนักรายมิติโดยตรง ใช้ C (ส่วนกลับของ regularization) — ต้องการข้อมูลมากกว่า (ตาราง 3.47)
    ถ้าผู้ใช้มีตัวอย่างไม่ครบทั้งสองคลาส หรือน้อยกว่า min_examples จะ "ฝึกไม่ได้" → score() คืน None
    ให้ผู้เรียกใช้โปรไฟล์ centroid แทน (บันได 3.48: ระดับ 1–2 ก่อน ระดับ 3)
    """
    def __init__(self, kind="nb", alpha=1.0, C=1.0, like_thr=4, dislike_thr=2, min_examples=2, seed=SEED):
        self.kind, self.alpha, self.C = kind, alpha, C
        self.like_thr, self.dislike_thr, self.min_examples, self.seed = like_thr, dislike_thr, min_examples, seed
        self.model = None

    def fit(self, X, hist):
        pos = [i for i, r in hist.items() if r >= self.like_thr]
        neg = [i for i, r in hist.items() if r <= self.dislike_thr]
        if len(pos) < self.min_examples or len(neg) < 1:
            self.model = None
            return self
        idx = pos + neg
        y = np.array([1] * len(pos) + [0] * len(neg))
        Xs = X[idx]
        if self.kind == "nb":
            from sklearn.naive_bayes import MultinomialNB
            self.model = MultinomialNB(alpha=self.alpha).fit(Xs, y)
        else:
            from sklearn.linear_model import LogisticRegression
            self.model = LogisticRegression(C=self.C, max_iter=1000, random_state=self.seed).fit(Xs, y)
        return self

    def score(self, X):
        """ความน่าจะเป็น P(ชอบ | เนื้อหา) ของทุกรายการ หรือ None เมื่อฝึกไม่ได้"""
        if self.model is None:
            return None
        return self.model.predict_proba(X)[:, 1]


# =========================================================================================
# ส่วนที่ 6  ฟังก์ชันจับคู่ การจัดอันดับ Top-K และ MMR (หัวข้อ 3.9–3.10)
# =========================================================================================
def match_scores(X, profile):
    """คะแนนจับคู่ = ผลคูณจุดระหว่างทุกแถวของ X กับโปรไฟล์ (เมื่อทั้งคู่ผ่าน L2 = โคไซน์) คืน np.array ยาว n_items"""
    s = X @ profile
    return np.asarray(s).ravel()


def item_knn_scores(S, liked_idx, agg="mean", k=None):
    """Item-kNN เชิงเนื้อหา (3.9.5): เทียบผู้สมัครกับรายการที่ชอบ "ทีละรายการ" แล้วรวม
    S: เมทริกซ์ความคล้าย n×n ; agg='mean' เฉลี่ย, 'max' เอาค่าสูงสุด ; k = ใช้เฉพาะ k เพื่อนบ้านที่คล้ายที่สุด"""
    if not liked_idx:
        return np.zeros(S.shape[0])
    sims = S[:, liked_idx]                       # n × |liked|
    if k is not None and k < sims.shape[1]:
        sims = -np.sort(-sims, axis=1)[:, :k]
    return sims.mean(axis=1) if agg == "mean" else sims.max(axis=1)


def explain_terms(vectorizer, profile, item_vec, k=8):
    """อธิบายผล (3.9.4): คำใดทำให้คะแนนสูง = ส่วนร่วมรายมิติ p_j · v_j เรียงจากมากไปน้อย"""
    v = to_dense_row(item_vec)
    contrib = profile * v
    order = np.argsort(-contrib)[:k]
    return [(vectorizer.vocab_[j], round(float(profile[j]), 4), round(float(v[j]), 4), round(float(contrib[j]), 4))
            for j in order if contrib[j] > 0]


def top_k(scores, seen, K=10, min_score=None, item_ids=None, candidates=None):
    """ห้าขั้นของ 3.10.1: รวบรวมผู้สมัคร → (คะแนนมีแล้ว) → กรองที่เคยเห็น → เรียง+ตัดสินเสมอ → เกณฑ์ขั้นต่ำ → ตัด K
    seen       : เซตดัชนีรายการที่ผู้ใช้เคยมีปฏิสัมพันธ์ในชุดฝึก (ห้ามแนะนำซ้ำ)
    candidates : เซตดัชนีผู้สมัคร (None = ทั้งคลัง = full-catalog ranking นิยาม 2.7)
    ตัดสินเสมอ : คะแนนเท่ากันเรียงตามดัชนี/รหัสรายการ (กฎที่ทำซ้ำได้ของบท — 3.10.2) ผ่าน np.lexsort
    คืน list ดัชนี (ยาว ≤ K)"""
    n = len(scores)
    cand = np.arange(n) if candidates is None else np.asarray(sorted(candidates))
    if seen:
        cand = cand[~np.isin(cand, list(seen))]
    if min_score is not None:
        cand = cand[scores[cand] >= min_score]
    if len(cand) == 0:
        return []
    key2 = cand if item_ids is None else np.asarray([item_ids[i] for i in cand])
    order = np.lexsort((key2, -scores[cand]))     # เรียงคะแนนมาก→น้อย แล้วรหัสน้อย→มากเมื่อเสมอ
    return [int(i) for i in cand[order][:K]]


def mmr_rerank(scores, S, K=10, lam=0.7, seen=None, pool=None, min_score=None, normalize=True):
    """Maximal Marginal Relevance (นิยาม 3.8): เลือกทีละรายการแบบละโมบ
        argmax_i  λ·rel(i) − (1−λ)·max_{j∈เลือกแล้ว} sim(i, j)
    λ=1 → เรียงตามความเกี่ยวข้องล้วน ; λ→0 → เน้นความหลากหลาย ; ค่านิยม 0.7 (ตัวอย่าง 3.9)
    pool: จำนวนผู้สมัครต้นทางที่นำมาจัดอันดับซ้ำ (ค่าตั้งต้น = 5K) เพื่อประหยัดเวลาในคลังใหญ่
    normalize=True จะปรับ rel ด้วย min-max ให้อยู่ในมาตรา 0–1 เดียวกับ sim (จำเป็นเมื่อคะแนนเป็น BM25/RRF)
    normalize=False ใช้คะแนนดิบ (ตรงกับตัวอย่างการคำนวณ 3.9 ในตำรา ซึ่ง rel อยู่ในช่วง 0–1 อยู่แล้ว)"""
    pool = pool or 5 * K
    base = top_k(scores, seen or set(), K=pool, min_score=min_score)
    if not base:
        return []
    rel = scores[base]
    if normalize and rel.max() > rel.min():
        rel = (rel - rel.min()) / (rel.max() - rel.min())
    chosen, remaining = [], list(range(len(base)))
    while remaining and len(chosen) < K:
        best, best_val = None, -1e18
        for r in remaining:
            div = max(S[base[r], base[c]] for c in chosen) if chosen else 0.0
            val = lam * rel[r] - (1 - lam) * div
            if val > best_val:
                best, best_val = r, val
        chosen.append(best); remaining.remove(best)
    return [base[c] for c in chosen]


# =========================================================================================
# ส่วนที่ 7  ผสานคะแนน · ปรับเทียบ (หัวข้อ 3.18)
# =========================================================================================
def minmax(x):
    """ปรับมาตรา 0–1 (ข้อบังคับก่อนผสมแบบนูน — ภาพที่ 3.32) ; ถ้าค่าคงที่คืนศูนย์ทั้งหมด"""
    x = np.asarray(x, dtype=np.float64)
    lo, hi = np.nanmin(x), np.nanmax(x)
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)


def zscore(x):
    x = np.asarray(x, dtype=np.float64)
    s = x.std()
    return (x - x.mean()) / s if s > 0 else np.zeros_like(x)


def convex_fusion(score_list, weights=None, scale="minmax"):
    """ผสมแบบนูน: Σ w_k · scale(s_k) โดย Σw = 1 (ตัวอย่าง 3.22) — ผู้ชนะเปลี่ยนตาม w จึงต้องปรับ w บนชุดตรวจสอบ"""
    m = len(score_list)
    w = np.ones(m) / m if weights is None else np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    f = minmax if scale == "minmax" else zscore
    return sum(wi * f(s) for wi, s in zip(w, score_list))


def rrf_fusion(score_list, k=60, weights=None):
    """Reciprocal Rank Fusion: Σ 1/(k + rank_i) ใช้เฉพาะ "อันดับ" จึงไม่ต้องปรับมาตรา (k=60 ตามธรรมเนียม)"""
    n = len(score_list[0])
    out = np.zeros(n)
    w = np.ones(len(score_list)) if weights is None else np.asarray(weights)
    for wi, s in zip(w, score_list):
        ranks = np.empty(n)
        ranks[np.argsort(-np.asarray(s), kind="stable")] = np.arange(1, n + 1)
        out += wi / (k + ranks)
    return out


class PlattCalibrator:
    """Platt scaling (นิยาม 3.19): p = σ(a·s + b) ประมาณ a, b จากชุดตรวจสอบด้วย log-loss ต่ำสุด
    (ใช้ LogisticRegression 1 มิติ) — ห้ามใช้ชุดทดสอบปรับเทียบ (รั่วไหลแบบที่ 5 ในตาราง 2.10)"""
    def fit(self, scores, labels):
        from sklearn.linear_model import LogisticRegression
        self.lr = LogisticRegression(C=1e6, max_iter=1000).fit(np.asarray(scores).reshape(-1, 1), labels)
        self.a_, self.b_ = float(self.lr.coef_[0, 0]), float(self.lr.intercept_[0])
        return self
    def predict_proba(self, scores):
        return self.lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]


def expected_calibration_error(probs, labels, n_bins=10):
    """ECE (นิยาม 3.20): แบ่งถังตามความมั่นใจ แล้วเฉลี่ย |สัดส่วนถูกจริง − ความมั่นใจเฉลี่ย| ถ่วงด้วยขนาดถัง
    คืน (ece, ตารางรายถัง) — ตัวอย่าง 3.23 ใช้ 5 ถัง"""
    probs, labels = np.asarray(probs), np.asarray(labels)
    edges = np.linspace(0, 1, n_bins + 1)
    rows, ece = [], 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (probs > lo) & (probs <= hi) if lo > 0 else (probs >= lo) & (probs <= hi)
        if m.sum() == 0:
            continue
        acc, conf = labels[m].mean(), probs[m].mean()
        ece += m.sum() / len(probs) * abs(acc - conf)
        rows.append({"ถัง": f"({lo:.1f}, {hi:.1f}]", "จำนวน": int(m.sum()), "สัดส่วนถูกจริง": round(acc, 4),
                     "ความมั่นใจเฉลี่ย": round(conf, 4), "|ผลต่าง|": round(abs(acc - conf), 4)})
    return ece, pd.DataFrame(rows)


# =========================================================================================
# ส่วนที่ 8  โมเดลทำนายคะแนน (Rating Prediction) — เพื่อวัด MAE/RMSE ของบทที่ 2 (2.1)
# =========================================================================================
class ContentKNNRegressor:
    """ทำนายคะแนน r̂(u,i) จากรายการที่ u เคยให้คะแนน ถ่วงด้วยความคล้าย "เชิงเนื้อหา" ระหว่างรายการ
        r̂(u,i) = μ_u + Σ_j sim(i,j)·(r_uj − μ_u) / Σ_j |sim(i,j)|     (j = k เพื่อนบ้านที่คล้ายที่สุดที่ u เคยให้คะแนน)
    * ไม่ต้องใช้ผู้ใช้คนอื่นเลย (ข้อดีข้อแรกของ content-based 3.19.1)
    * ถ้าไม่มีเพื่อนบ้าน (sim=0 ทุกคู่) จะถอยไปใช้ค่าเฉลี่ยของผู้ใช้ μ_u ; ถ้าผู้ใช้ใหม่ใช้ค่าเฉลี่ยรวม μ
    * shrink: ลดอิทธิพลเมื่อผลรวมความคล้ายเล็ก (กัน overfit เพื่อนบ้านน้อย)
    """
    def __init__(self, X, k=20, shrink=0.0, min_sim=0.0):
        self.Xn = l2_normalize_rows(X)
        self.k, self.shrink, self.min_sim = k, shrink, min_sim

    def fit(self, train):
        """train: DataFrame มีคอลัมน์ u (ดัชนีผู้ใช้), i (ดัชนีรายการ), r (คะแนน) — ต้องเป็น "ชุดฝึกเท่านั้น" """
        self.mu = float(train["r"].mean())
        self.user_hist = {u: (g["i"].to_numpy(), g["r"].to_numpy()) for u, g in train.groupby("u")}
        self.user_mean = {u: float(r.mean()) for u, (_, r) in self.user_hist.items()}
        return self

    def predict_one(self, u, i):
        if u not in self.user_hist:
            return self.mu
        items, ratings = self.user_hist[u]
        mu_u = self.user_mean[u]
        sims = np.asarray((self.Xn[items] @ self.Xn[i].T).todense()).ravel() if sp.issparse(self.Xn) \
            else self.Xn[items] @ self.Xn[i]
        m = sims > self.min_sim
        if m.sum() == 0:
            return mu_u
        sims, ratings = sims[m], ratings[m]
        if self.k and len(sims) > self.k:
            top = np.argsort(-sims)[: self.k]
            sims, ratings = sims[top], ratings[top]
        den = np.abs(sims).sum() + self.shrink
        return mu_u + float((sims * (ratings - mu_u)).sum() / den) if den > 0 else mu_u

    def predict(self, pairs):
        """pairs: DataFrame คอลัมน์ u, i → np.array คะแนนทำนาย (ผู้เรียกควร clip ให้อยู่ในช่วงคะแนนเอง)
        คำนวณเป็นกลุ่มต่อผู้ใช้ (เมทริกซ์ความคล้าย |ประวัติ| × |รายการที่ต้องทำนาย|) เร็วกว่าเรียก predict_one ทีละคู่มาก"""
        out = np.full(len(pairs), self.mu)
        pu, pi = pairs["u"].to_numpy(), pairs["i"].to_numpy()
        for u in np.unique(pu):
            m = np.where(pu == u)[0]
            if u not in self.user_hist:
                continue
            items, ratings = self.user_hist[u]; mu_u = self.user_mean[u]
            S = self.Xn[items] @ self.Xn[pi[m]].T
            S = np.asarray(S.todense()) if sp.issparse(S) else np.asarray(S)      # |hist| × |targets|
            S[S <= self.min_sim] = 0.0
            if self.k and S.shape[0] > self.k:                                   # เก็บเฉพาะ k เพื่อนบ้านที่คล้ายที่สุดต่อคอลัมน์
                thr = -np.partition(-S, self.k - 1, axis=0)[self.k - 1]
                S = np.where(S >= thr, S, 0.0)
            den = np.abs(S).sum(axis=0) + self.shrink
            num = (S * (ratings - mu_u)[:, None]).sum(axis=0)
            pred = np.where(den > 0, mu_u + num / np.where(den > 0, den, 1.0), mu_u)
            pred = np.where(np.abs(S).sum(axis=0) > 0, pred, mu_u)
            out[m] = pred
        return out


class MeanRatingBaseline:
    """ระบบฐานสำหรับ RMSE/MAE: 'global' = ค่าเฉลี่ยรวม, 'user' = ค่าเฉลี่ยผู้ใช้, 'item' = ค่าเฉลี่ยรายการ,
    'user_item' = μ + b_u + b_i (baseline ของ Koren) — ระบบใหม่ที่แพ้ค่าเฉลี่ยผู้ใช้คือสัญญาณเตือน"""
    def __init__(self, kind="user"):
        self.kind = kind
    def fit(self, train):
        self.mu = float(train["r"].mean())
        self.bu = (train.groupby("u")["r"].mean() - self.mu).to_dict()
        self.bi = (train.groupby("i")["r"].mean() - self.mu).to_dict()
        return self
    def predict(self, pairs):
        out = []
        for u, i in zip(pairs["u"], pairs["i"]):
            if self.kind == "global":
                out.append(self.mu)
            elif self.kind == "user":
                out.append(self.mu + self.bu.get(u, 0.0))
            elif self.kind == "item":
                out.append(self.mu + self.bi.get(i, 0.0))
            else:
                out.append(self.mu + self.bu.get(u, 0.0) + self.bi.get(i, 0.0))
        return np.array(out)


# =========================================================================================
# ส่วนที่ 9  ตัวชี้วัดของบทที่ 2
# =========================================================================================
# ---- 2.1 ตัวชี้วัดการทำนายคะแนน ----
def mae(y_true, y_pred):
    """Mean Absolute Error (2.1.1) — ทุกความผิดพลาดหนักเท่ากัน"""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true, y_pred):
    """Root Mean Squared Error (2.1.2) — ลงโทษความผิดพลาดใหญ่มากกว่า (ตัวอย่าง 2.2: MAE เท่ากันแต่ RMSE ต่างกัน)"""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


# ---- 2.2–2.3 ตัวชี้วัดเชิงอันดับ "รายผู้ใช้" (rec_list = ลำดับรายการที่แนะนำ, rel = เซตรายการที่เกี่ยวข้อง) ----
def precision_at_k(rec, rel, K):
    """P@K = |Top-K ∩ Rel| / K  (2.2.1) ตัวหารคือ K เสมอ แม้ผู้ใช้จะมี Rel น้อยกว่า K"""
    return len(set(rec[:K]) & rel) / K


def recall_at_k(rec, rel, K):
    """R@K = |Top-K ∩ Rel| / |Rel|  (2.2.2) ; ผู้ใช้ที่ไม่มี Rel เลยควรถูกตัดออกก่อนประเมิน"""
    return len(set(rec[:K]) & rel) / len(rel) if rel else 0.0


def f1_at_k(rec, rel, K):
    p, r = precision_at_k(rec, rel, K), recall_at_k(rec, rel, K)
    return 2 * p * r / (p + r) if p + r > 0 else 0.0


def hit_at_k(rec, rel, K):
    """HR@K = 1 ถ้ามีรายการเกี่ยวข้องอย่างน้อยหนึ่งใน Top-K (2.2.4) — นิยมกับ leave-one-out"""
    return 1.0 if set(rec[:K]) & rel else 0.0


def rr_at_k(rec, rel, K):
    """Reciprocal Rank (2.3.1): 1/อันดับของรายการเกี่ยวข้อง "ตัวแรก" ; 0 ถ้าไม่มีใน K"""
    for pos, it in enumerate(rec[:K], start=1):
        if it in rel:
            return 1.0 / pos
    return 0.0


def ap_at_k(rec, rel, K):
    """Average Precision@K (2.3.2): เฉลี่ย P@i เฉพาะตำแหน่ง i ที่เป็นรายการเกี่ยวข้อง หารด้วย min(|Rel|, K)"""
    if not rel:
        return 0.0
    hits, s = 0, 0.0
    for pos, it in enumerate(rec[:K], start=1):
        if it in rel:
            hits += 1
            s += hits / pos
    return s / min(len(rel), K)


def dcg_at_k(gains):
    """DCG = Σ gain_i / log2(i+1) (2.3.3) — ตำแหน่ง 1 หารด้วย log2(2)=1, ตำแหน่ง 2 หารด้วย log2(3)…"""
    return float(sum(g / math.log2(i + 1) for i, g in enumerate(gains, start=1)))


def ndcg_at_k(rec, rel, K, rel_grades=None, gain_type="linear"):
    """nDCG@K = DCG@K / IDCG@K
    rel_grades : dict item→ระดับความเกี่ยวข้อง (เช่น rating) ถ้า None ใช้ 1/0 (binary)
    gain_type  : 'linear' gain=grade ; 'exp' gain=2^grade−1 (ต้องประกาศแบบที่ใช้ — ตาราง 2.16 ขั้น 4)"""
    def g(it):
        grade = (rel_grades.get(it, 0.0) if rel_grades else (1.0 if it in rel else 0.0))
        if it not in rel:
            grade = 0.0
        return (2 ** grade - 1) if gain_type == "exp" else grade
    gains = [g(it) for it in rec[:K]]
    ideal = sorted([g(it) for it in rel], reverse=True)[:K]
    idcg = dcg_at_k(ideal)
    return dcg_at_k(gains) / idcg if idcg > 0 else 0.0


def auc_user(scores, pos_idx, neg_idx):
    """AUC (นิยาม 2.1) นับคู่ทีละคู่: สัดส่วนคู่ (บวก, ลบ) ที่รายการบวกได้คะแนนสูงกว่า (เสมอนับ 0.5)"""
    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return np.nan
    ps, ns = scores[list(pos_idx)], scores[list(neg_idx)]
    diff = ps[:, None] - ns[None, :]
    return float(((diff > 0).sum() + 0.5 * (diff == 0).sum()) / diff.size)


# ---- 2.5 ตัวชี้วัด Beyond-Accuracy (ระดับระบบ / ระดับรายการแนะนำ) ----
def catalog_coverage(rec_lists, n_items):
    """Coverage@K (2.5.1) = |⋃_u Top-K(u)| / |I| — รายการนับครั้งเดียวไม่ว่าจะโผล่กี่คน"""
    s = set()
    for r in rec_lists:
        s.update(r)
    return len(s) / n_items


def gini_exposure(rec_lists, n_items):
    """Gini ของการเปิดเผย (2.5.2): x_i = จำนวนครั้งที่รายการ i ถูกแนะนำ "เรียงจากน้อยไปมาก" (ขั้นบังคับ)
        G = Σ (2i − n − 1) x_i / (n Σ x_i)   ; 0 = เท่าเทียม, →1 = กระจุกที่รายการเดียว
    รายการที่ไม่เคยถูกแนะนำนับเป็น x=0 ด้วย (n = ทั้งคลัง)"""
    x = np.zeros(n_items)
    for r in rec_lists:
        for it in r:
            x[it] += 1
    x = np.sort(x)
    n = len(x)
    if x.sum() == 0:
        return 0.0
    i = np.arange(1, n + 1)
    return float(((2 * i - n - 1) * x).sum() / (n * x.sum()))


def head_items_from_train(train_item_counts, n_items, share=1 / 3):
    """เส้นแบ่งหัว–หาง (นิยาม 2.2, ตัวอย่าง 2.8): เรียงตามความนิยม "จากชุดฝึก" แล้วสะสมจนถึง share ของปฏิสัมพันธ์ทั้งหมด
    คืน set ของ head items (ที่เหลือ = long tail) — ต้องประกาศเกณฑ์ที่ใช้ทุกครั้ง และห้ามคำนวณจากข้อมูลทั้งหมด (รั่ว)"""
    counts = np.zeros(n_items)
    for i, c in train_item_counts.items():
        counts[i] = c
    order = np.argsort(-counts, kind="stable")
    cum, total, head = 0.0, counts.sum(), set()
    for i in order:
        if counts[i] == 0:
            break
        head.add(int(i)); cum += counts[i]
        if cum >= share * total:
            break
    return head


def long_tail_ratio(rec, head, K):
    """LongTailRatio@K = สัดส่วนรายการใน Top-K ที่ "ไม่ใช่" head"""
    top = rec[:K]
    return sum(1 for it in top if it not in head) / K if top else 0.0


def intra_list_diversity(rec, F, K):
    """ILD@K (นิยาม 2.3) = 2/(K(K−1)) Σ_{i<j} (1 − cos(f_i, f_j)) วัดจากคุณลักษณะรายการ F (เช่น เวกเตอร์แนว)
    ค่าเข้าใกล้ 1 = ต่างกันมาก ; รายการที่มีชิ้นเดียวคืน 0"""
    top = rec[:K]
    if len(top) < 2:
        return 0.0
    M = l2_normalize_rows(F[top])
    S = M @ M.T
    S = np.asarray(S.todense()) if sp.issparse(S) else np.asarray(S)
    n = len(top)
    iu = np.triu_indices(n, k=1)
    return float((1 - S[iu]).mean())


def novelty(rec, p_item, K):
    """Novelty@K (2.5.4) = (1/K) Σ −log2 p(i) ; p(i) = "สัดส่วนผู้ใช้" ในชุดฝึกที่เคยมีปฏิสัมพันธ์กับ i (หน่วยบิต)
    รายการที่ไม่มีใครเคยเห็นเลย p=0 → ใช้ p = 1/(จำนวนผู้ใช้+1) กัน log(0) (ต้องประกาศ)"""
    top = rec[:K]
    if not top:
        return 0.0
    return float(np.mean([-math.log2(max(p_item[it], p_item.min_p)) for it in top]))


class ItemPopularity:
    """เก็บ p(i) = สัดส่วนผู้ใช้ที่เคยมีปฏิสัมพันธ์กับ i (คำนวณจากชุดฝึกเท่านั้น) และจำนวนปฏิสัมพันธ์รายรายการ"""
    def __init__(self, train, n_items, n_users):
        c_users = train.groupby("i")["u"].nunique()
        self.p = np.zeros(n_items)
        self.p[c_users.index.to_numpy()] = c_users.to_numpy() / n_users
        self.count = train.groupby("i").size().to_dict()
        self.min_p = 1.0 / (n_users + 1)
    def __getitem__(self, i):
        return self.p[i]


def calibration_error(hist_items, rec, G, K):
    """Calibration (2.5.5): 0.5 Σ_c |q(c) − p(c)| ; p = สัดส่วนแนวในประวัติ, q = สัดส่วนแนวในรายการแนะนำ
    G = เมทริกซ์รายการ×แนว (0/1) ; ค่าใกล้ 0 = สอดคล้องกับประวัติ"""
    def dist(idx):
        if len(idx) == 0:
            return None
        v = np.asarray(G[list(idx)].sum(axis=0)).ravel().astype(float)
        return v / v.sum() if v.sum() > 0 else v
    p, q = dist(hist_items), dist(rec[:K])
    if p is None or q is None:
        return np.nan
    return float(0.5 * np.abs(q - p).sum())


def parity_gap(per_user_df, metric, group_col):
    """Fairness ฝั่งผู้ใช้ (2.5.6): ผลต่างของ Primary Metric ระหว่างกลุ่มผู้ใช้ (เช่น ประวัติยาว vs สั้น)"""
    g = per_user_df.groupby(group_col)[metric].mean()
    return float(g.max() - g.min()), g


# ---- ตัวประเมินหลัก: จัดอันดับ + คำนวณทุกตัวชี้วัดในรอบเดียว ----
def evaluate_ranking(score_fn, users, train_hist, test_rel, n_items, K=10, item_feat=None, pop=None,
                     head=None, genre_mat=None, rel_grades=None, candidate="full", m_neg=99, seed=SEED,
                     rerank_fn=None, min_score=None, item_ids=None, pool=None):
    """ประเมินโมเดลจัดอันดับหนึ่งตัวบนผู้ใช้ทุกคน คืน (ตารางรายผู้ใช้, สรุปค่าเฉลี่ย, dict รายการแนะนำ)

    score_fn(u)  → np.array คะแนนทุกรายการ (ดัชนี 0..n_items-1) ของผู้ใช้ u ; คืน None = ไม่มีหลักฐาน
    train_hist   : dict u → {item_index: rating} จากชุดฝึก (ใช้กรองที่เคยเห็น และ calibration)
    test_rel     : dict u → set(item_index) ที่เกี่ยวข้องในชุดที่กันไว้ (เช่น rating ≥ 4)
    candidate    : 'full' = จัดอันดับทั้งคลังที่ยังไม่เห็น (นิยาม 2.7) ; 'sampled' = เป้าหมาย + m_neg รายการลบสุ่ม
                   (ต้องรายงานว่าเป็น sampled metric และห้ามเทียบข้าม m — 2.9.5) ; รายการลบไม่ถูกวางไว้ตำแหน่งแรก
    rerank_fn(u, scores) → list ดัชนี (เช่น MMR) ใช้แทนการเรียงตามคะแนนเมื่อกำหนด
    item_feat    : เมทริกซ์คุณลักษณะสำหรับ ILD (เช่น เวกเตอร์แนว) ; genre_mat สำหรับ calibration
    pop          : ItemPopularity สำหรับ Novelty ; head : เซต head items สำหรับ LongTail
    pool         : เซตดัชนีรายการที่ "ให้บริการได้" (servable catalog เช่น รายการที่อยู่ใน k-core ของชุดฝึก)
                   None = ทั้งคลัง ; เมื่อกำหนด ทั้ง candidate, AUC และตัวหารของ Coverage/Gini จะใช้เซตนี้ (ต้องประกาศ)
    ผู้ใช้ที่ไม่มีรายการเกี่ยวข้องในชุดทดสอบจะถูกข้าม (ประกาศจำนวนที่ประเมินจริงเสมอ)
    """
    rng = np.random.default_rng(seed)
    rows, rec_lists, recs = [], [], {}
    pool_arr = np.arange(n_items) if pool is None else np.asarray(sorted(pool))
    pool_set = None if pool is None else set(int(i) for i in pool_arr)
    n_cat = len(pool_arr)
    for u in users:
        rel = test_rel.get(u, set())
        if not rel:
            continue
        seen = set(train_hist.get(u, {}).keys())
        s = score_fn(u)
        if s is None:
            s = np.zeros(n_items)
        s = np.asarray(s, dtype=np.float64)
        cands = pool_set
        if candidate == "sampled":
            unseen = np.setdiff1d(pool_arr, list(seen | rel))
            neg = rng.choice(unseen, size=min(m_neg, len(unseen)), replace=False)
            cands = set(neg.tolist()) | rel
        if rerank_fn is not None:
            rec = rerank_fn(u, s)
        else:
            rec = top_k(s, seen, K, min_score=min_score, item_ids=item_ids, candidates=cands)
        recs[u] = rec
        rec_lists.append(rec)
        row = {"user": u, "n_rel": len(rel), "n_hist": len(seen),
               "P@K": precision_at_k(rec, rel, K), "R@K": recall_at_k(rec, rel, K), "F1@K": f1_at_k(rec, rel, K),
               "HR@K": hit_at_k(rec, rel, K), "RR@K": rr_at_k(rec, rel, K), "AP@K": ap_at_k(rec, rel, K),
               "nDCG@K": ndcg_at_k(rec, rel, K, rel_grades.get(u) if rel_grades else None)}
        if candidate == "full":
            neg_pool = np.setdiff1d(pool_arr, list(seen | rel))
            if len(neg_pool) > 500:
                neg_pool = rng.choice(neg_pool, 500, replace=False)   # ประมาณ AUC ด้วยตัวอย่างลบ 500 ชิ้น
            row["AUC"] = auc_user(s, list(rel), neg_pool)
        if item_feat is not None:
            row["ILD@K"] = intra_list_diversity(rec, item_feat, K)
        if pop is not None:
            row["Novelty@K"] = novelty(rec, pop, K)
        if head is not None:
            row["LongTail@K"] = long_tail_ratio(rec, head, K)
        if genre_mat is not None:
            row["CalibErr"] = calibration_error(list(seen), rec, genre_mat, K)
        rows.append(row)
    per_user = pd.DataFrame(rows)
    summary = per_user.drop(columns=["user", "n_rel", "n_hist"]).mean(numeric_only=True).to_dict()
    summary["MRR@K"] = summary.pop("RR@K", 0.0)
    summary["MAP@K"] = summary.pop("AP@K", 0.0)
    summary["Coverage@K"] = catalog_coverage(rec_lists, n_cat)      # ตัวหาร = ขนาดคลังที่ให้บริการได้ (pool) หรือทั้งคลัง
    summary["Gini"] = gini_exposure(rec_lists, n_items) if pool is None else gini_exposure(
        [[int(np.searchsorted(pool_arr, it)) for it in r if it in pool_set] for r in rec_lists], n_cat)
    summary["n_users"] = len(per_user)
    return per_user, summary, recs


# =========================================================================================
# ส่วนที่ 10  การออกแบบการทดลองออฟไลน์ (หัวข้อ 2.7–2.10, 2.14)
# =========================================================================================
def encode_ids(df, user_col="userId", item_col="movieId", items_df=None, item_id_col="movieId"):
    """แปลงรหัสผู้ใช้/รายการเป็นดัชนี 0..n-1 (คอลัมน์ u, i) — รายการอ้างอิงตาม items_df เพื่อให้คลังครบทุกชิ้น
    แม้ไม่มีใครให้คะแนน (จำเป็นต่อ Coverage ที่ตัวหารคือ "ทั้งคลัง")"""
    users = sorted(df[user_col].unique())
    u_map = {u: k for k, u in enumerate(users)}
    if items_df is not None:
        items = list(items_df[item_id_col])
    else:
        items = sorted(df[item_col].unique())
    i_map = {it: k for k, it in enumerate(items)}
    out = df.copy()
    out["u"] = out[user_col].map(u_map)
    out["i"] = out[item_col].map(i_map)
    out = out.dropna(subset=["u", "i"]).astype({"u": int, "i": int})
    if "rating" in out.columns:
        out["r"] = out["rating"].astype(float)
    return out, u_map, i_map


def kcore_filter(df, k_user=5, k_item=5, max_iter=20):
    """กรอง k-core "ก่อน" แบ่งข้อมูล (รูปแบบรั่วไหลที่ 3 ในตาราง 2.10 คือกรองหลังแบ่ง) — วนจนนิ่ง"""
    d = df
    for _ in range(max_iter):
        n0 = len(d)
        uc = d.groupby("u").size(); ic = d.groupby("i").size()
        d = d[d["u"].isin(uc[uc >= k_user].index) & d["i"].isin(ic[ic >= k_item].index)]
        if len(d) == n0:
            break
    return d


def split_random(df, test_frac=0.2, seed=SEED):
    """แบบที่ 1 สุ่มโดยไม่สนเวลา — ง่ายแต่ "ใช้อนาคตทำนายอดีต" เมื่อข้อมูลมีเวลากำกับ (2.8.1)"""
    rng = np.random.default_rng(seed)
    m = rng.random(len(df)) < test_frac
    return df[~m].copy(), df[m].copy()


def split_global_temporal(df, test_frac=0.2, time_col="timestamp"):
    """แบบที่ 2 จุดตัดเวลาจุดเดียวทั้งระบบ: ก่อน = ฝึก หลัง = ทดสอบ (จำลองการใช้งานจริงตรงที่สุด)
    ข้อสังเกต: ผู้ใช้บางคนอาจไม่มีข้อมูลฝึกเลย (ผู้ใช้ใหม่จริง ๆ) — ต้องตัดสินใจว่าจะประเมินหรือไม่และประกาศ"""
    cut = df[time_col].quantile(1 - test_frac)
    return df[df[time_col] <= cut].copy(), df[df[time_col] > cut].copy()


def split_per_user_temporal(df, test_frac=0.2, time_col="timestamp", min_train=1):
    """แบบที่ 3 กันส่วนท้าย test_frac ของ "แต่ละคน" — ทุกคนมีชุดทดสอบ แต่จุดตัดไม่ตรงกัน (ยังรั่วข้ามผู้ใช้ได้)"""
    tr, te = [], []
    for u, g in df.sort_values(time_col).groupby("u"):
        n_te = int(np.ceil(len(g) * test_frac))
        n_te = min(n_te, max(0, len(g) - min_train))
        if n_te == 0:
            tr.append(g); continue
        tr.append(g.iloc[:-n_te]); te.append(g.iloc[-n_te:])
    return pd.concat(tr), (pd.concat(te) if te else df.iloc[0:0].copy())


def split_leave_one_out(df, time_col="timestamp", only_relevant=False, like_thr=4, seed=SEED):
    """แบบที่ 4 กันเหตุการณ์ "ล่าสุด" ของแต่ละคนไว้หนึ่งชิ้น (ไม่มีเวลา → สุ่มหนึ่งชิ้น)
    only_relevant=True: กันเฉพาะรายการที่ชอบ (rating ≥ like_thr) เพื่อให้ทุกผู้ใช้มีเป้าหมายวัด HR/MRR ได้
    ความแปรปรวนสูงมากเพราะตัดสินผู้ใช้ด้วยเหตุการณ์เดียว (2.8.1)"""
    rng = np.random.default_rng(seed)
    tr, te = [], []
    for u, g in df.groupby("u"):
        pool = g[g["r"] >= like_thr] if only_relevant and "r" in g else g
        if len(g) < 2 or len(pool) == 0:
            tr.append(g); continue
        if time_col in g.columns:
            hold = pool.sort_values(time_col).index[-1]
        else:
            hold = rng.choice(pool.index)
        te.append(g.loc[[hold]]); tr.append(g.drop(index=hold))
    return pd.concat(tr), pd.concat(te)


def three_way_split(df, protocol="global_temporal", valid_frac=0.1, test_frac=0.2, seed=SEED, **kw):
    """แบ่งสามชุด Train → Validation → Test (ตาราง 2.10 แถวที่ 5: เลือก hyperparameter บน validation เท่านั้น
    และแตะ test ครั้งเดียวตอนท้าย) โดยใช้โปรโตคอลเดียวกันสองครั้ง: ตัด test ออกก่อน แล้วตัด valid จากที่เหลือ"""
    f = {"random": split_random, "global_temporal": split_global_temporal,
         "per_user_temporal": split_per_user_temporal}[protocol]
    if protocol == "random":
        rest, test = f(df, test_frac, seed=seed)
        train, valid = f(rest, valid_frac / (1 - test_frac), seed=seed + 1)
    else:
        rest, test = f(df, test_frac, **kw)
        train, valid = f(rest, valid_frac / (1 - test_frac), **kw)
    return train, valid, test


def leakage_check(train, test, time_col="timestamp"):
    """ตรวจรั่วไหลเชิงเวลา (2.8.4): นับเหตุการณ์ในชุดฝึกที่เกิด "หลัง" เหตุการณ์แรกของชุดทดสอบ
    0 = ปลอดภัยแบบ global temporal ; >0 = มีการรั่วข้ามเวลา (ยอมรับได้เฉพาะเมื่อประกาศ เช่น per-user split)"""
    if time_col not in train.columns or len(test) == 0:
        return {"future_events_in_train": None}
    t0 = test[time_col].min()
    n_future = int((train[time_col] > t0).sum())
    return {"future_events_in_train": n_future, "share": n_future / max(1, len(train)),
            "train_max_time": pd.to_datetime(train[time_col].max(), unit="s"),
            "test_min_time": pd.to_datetime(t0, unit="s")}


def hist_from(df):
    """dict u → {i: r} จาก DataFrame (ใช้กับชุดฝึก)"""
    return {u: dict(zip(g["i"], g["r"])) for u, g in df.groupby("u")}


def relevant_from(df, like_thr=4.0):
    """dict u → set(i) ที่ rating ≥ like_thr (นิยามความเกี่ยวข้อง — ต้องประกาศ ขั้นที่ 4 ของโปรโตคอล)
    และ dict u → {i: r} สำหรับ nDCG แบบ graded"""
    rel, grades = {}, {}
    for u, g in df.groupby("u"):
        m = g["r"] >= like_thr
        rel[u] = set(g.loc[m, "i"])
        grades[u] = dict(zip(g.loc[m, "i"], g.loc[m, "r"]))
    return rel, grades


def paired_bootstrap(a, b, n_boot=2000, seed=SEED, alpha=0.05):
    """Paired bootstrap ระดับผู้ใช้ (2.10.2): สุ่มซ้ำ "ผู้ใช้" (ไม่ใช่ปฏิสัมพันธ์ — ตัวอย่าง 2.17) แล้วดู CI ของผลต่าง b−a
    คืน dict {diff, ci_low, ci_high, p_value(สองด้าน โดยประมาณ), significant}"""
    a, b = np.asarray(a, float), np.asarray(b, float)
    rng = np.random.default_rng(seed)
    n = len(a)
    d = b - a
    boots = np.array([d[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return {"diff": float(d.mean()), "ci_low": float(lo), "ci_high": float(hi), "p_value": float(min(1.0, p)),
            "significant": bool(lo > 0 or hi < 0), "n_users": n}


def holm_bonferroni(pvals, alpha=0.05):
    """Holm–Bonferroni (2.10.4): เรียง p จากน้อยไปมาก เทียบกับ α/(m−k+1) ทีละขั้น หยุดเมื่อไม่ผ่านครั้งแรก
    คืน list ของ bool (ปฏิเสธ H0 หรือไม่) ตามลำดับเดิม"""
    m = len(pvals)
    order = np.argsort(pvals)
    reject = [False] * m
    for k, idx in enumerate(order):
        if pvals[idx] <= alpha / (m - k):
            reject[idx] = True
        else:
            break
    return reject


def seed_variance(run_fn, seeds, metric="nDCG@K"):
    """ความผันผวนจาก seed อย่างเดียว (2.10.1): run_fn(seed) → summary dict ; คืนตาราง + mean±sd
    หากผลต่างระหว่างสองโมเดลเล็กกว่า sd ของ seed ก็ยังสรุปไม่ได้ว่าใครดีกว่า"""
    rows = [{"seed": s, **run_fn(s)} for s in seeds]
    df = pd.DataFrame(rows)
    return df, float(df[metric].mean()), float(df[metric].std(ddof=1)) if len(df) > 1 else 0.0


class ProtocolCard(dict):
    """Evaluation Protocol Card (นิยาม 2.9 / ตาราง 2.15–2.16): บันทึกทุกข้อกำหนดของการทดลอง "ก่อน" ดูผล
    ใช้เป็น dict ธรรมดา แล้วเรียก .show() เพื่อพิมพ์เป็นตาราง 11 ขั้น"""
    FIELDS = ["1 เป้าหมายผู้ใช้/ธุรกิจ", "2 Primary Metric", "3 Guardrail Metrics", "4 สูตรและค่า K / นิยามความเกี่ยวข้อง",
              "5 การแบ่งข้อมูล", "6 การป้องกันรั่วไหล", "7 Candidate Set", "8 Negative Sampling",
              "9 Accuracy + Beyond-Accuracy", "10 ความไม่แน่นอน", "11 ตรวจกับผู้ใช้จริง"]
    def show(self):
        rows = [{"ขั้น": f, "ค่าที่กำหนด": self.get(f, "—")} for f in self.FIELDS]
        for k, v in self.items():
            if k not in self.FIELDS:
                rows.append({"ขั้น": k, "ค่าที่กำหนด": v})
        df = pd.DataFrame(rows)
        try:
            from IPython.display import display
            display(df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index"))
        except Exception:
            print(df.to_string(index=False))


# =========================================================================================
# ส่วนที่ 11  การนำเสนอผล ตาราง กราฟ รายงานผู้ใช้ และฟอร์ม GUI
# =========================================================================================
ACC_COLS = ["P@K", "R@K", "F1@K", "HR@K", "MRR@K", "MAP@K", "nDCG@K", "AUC"]
BEYOND_COLS = ["Coverage@K", "Gini", "LongTail@K", "ILD@K", "Novelty@K", "CalibErr"]


def results_table(summaries, cols=None, K=None):
    """รวม summary dict ของหลายโมเดลเป็นตารางเดียว (แถว = โมเดล) ; แทน @K ด้วยค่า K จริงเมื่อระบุ"""
    df = pd.DataFrame(summaries).T
    cols = cols or [c for c in ACC_COLS + BEYOND_COLS + ["n_users"] if c in df.columns]
    df = df[cols]
    if "n_users" in df.columns:
        df["n_users"] = df["n_users"].astype(int)
    if K is not None:
        df.columns = [c.replace("@K", f"@{K}") for c in df.columns]
    return df


def show_df(df, caption=None, fmt="{:.4f}"):
    """แสดง DataFrame แบบสวยใน notebook (fallback เป็น print เมื่อไม่มี IPython)"""
    try:
        from IPython.display import display
        num_cols = [c for c in df.columns if pd.api.types.is_float_dtype(df[c])]
        sty = df.style.format(fmt, na_rep="—", subset=num_cols) if num_cols else df.style
        if caption:
            sty = sty.set_caption(caption)
        display(sty)
    except Exception:
        if caption:
            print(caption)
        print(df.to_string())


def plot_metric_bars(table, metrics, title="", ncols=4, figsize=None, highlight_best=True):
    """กราฟแท่งเทียบโมเดล: หนึ่งกราฟย่อยต่อหนึ่งตัวชี้วัด (ระบายแท่งที่ดีที่สุดเข้มกว่า)
    ตัวชี้วัดที่ "ยิ่งน้อยยิ่งดี" (Gini, CalibErr, MAE, RMSE) จะทำเครื่องหมาย ↓ ในชื่อ"""
    import matplotlib.pyplot as plt
    metrics = [m for m in metrics if m in table.columns]
    n = len(metrics)
    if n == 0:
        return
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize or (4.2 * ncols, 3.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    lower_better = {"Gini", "CalibErr", "MAE", "RMSE"}
    for ax, m in zip(axes, metrics):
        vals = table[m].astype(float)
        lb = any(m.startswith(x) for x in lower_better)
        best = vals.idxmin() if lb else vals.idxmax()
        colors = ["#2E6B4A" if (highlight_best and idx == best) else "#9DBFA9" for idx in table.index]
        ax.bar(range(len(vals)), vals.values, color=colors)
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(table.index, rotation=45, ha="right", fontsize=8)
        ax.set_title(m + (" ↓" if lb else " ↑"), fontsize=10)
        for k, v in enumerate(vals.values):
            if not np.isnan(v):
                ax.text(k, v, f"{v:.3f}", ha="center", va="bottom", fontsize=7)
        ax.grid(axis="y", alpha=0.3)
    for ax in axes[n:]:
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


def plot_tradeoff(table, x="nDCG@K", y="ILD@K", title="Accuracy เทียบ Beyond-Accuracy"):
    """แผนภาพกระจาย: แกน x = accuracy, แกน y = beyond-accuracy — ดู "การแลกเปลี่ยน" (3.10.5, 3.20.1 ข้อ 9)"""
    import matplotlib.pyplot as plt
    if x not in table.columns or y not in table.columns:
        return
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for name, row in table.iterrows():
        ax.scatter(row[x], row[y], s=60, color="#2E6B4A")
        ax.annotate(name, (row[x], row[y]), textcoords="offset points", xytext=(5, 4), fontsize=8)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(title); ax.grid(alpha=0.3)
    plt.show()


def side_by_side_topk(recs_by_model, u, item_title, K=5, scores_by_model=None):
    """ตารางเทียบรายการแนะนำ Top-K ของผู้ใช้ u จากหลายโมเดล (คอลัมน์ = โมเดล) — ตัวอย่างผลลัพธ์บนชุดตรวจสอบ"""
    cols = {}
    for name, recs in recs_by_model.items():
        r = recs.get(u, [])[:K]
        cells = []
        for it in r:
            t = item_title(it)
            if scores_by_model and name in scores_by_model and scores_by_model[name].get(u) is not None:
                t += f" ({scores_by_model[name][u][it]:.3f})"
            cells.append(t)
        cells += [""] * (K - len(cells))
        cols[name] = cells
    return pd.DataFrame(cols, index=[f"อันดับ {k}" for k in range(1, K + 1)])


def mark_hits(df_topk, rel_titles):
    """ใส่เครื่องหมาย ✔ หน้ารายการที่อยู่ในของจริง (ground truth) เพื่ออ่านผลด้วยตา"""
    def f(x):
        if not x:
            return x
        base = re.sub(r" \(-?\d+\.\d+\)$", "", x)      # ตัดเฉพาะคะแนนท้ายข้อความ ไม่ตัดวงเล็บในชื่อเรื่อง เช่น "Toy Story (1995)"
        return ("✔ " if base in rel_titles else "   ") + x
    return df_topk.applymap(f) if hasattr(df_topk, "applymap") else df_topk.map(f)


def user_report(u, train_hist, test_df, recs_by_model, item_title, item_content=None, K=5, like_thr=4):
    """รายงานผู้ใช้หนึ่งคน 3 ส่วน: ① ประวัติ (ชุดฝึก) ② ของจริงในชุดที่กันไว้ ③ Top-K ของแต่ละโมเดล (✔ = ตรงของจริง)
    item_content(i) → str เนื้อหาของรายการ (ถ้ามี) ; คืน dict ของ DataFrame"""
    hist = train_hist.get(u, {})
    h = pd.DataFrame([{"item": item_title(i), "rating": r, **({"เนื้อหา": item_content(i)} if item_content else {})}
                      for i, r in sorted(hist.items(), key=lambda x: -x[1])])
    te = test_df[test_df["u"] == u]
    rel_titles = set(item_title(i) for i in te.loc[te["r"] >= like_thr, "i"])
    g = pd.DataFrame([{"item": item_title(i), "rating": r, "เกี่ยวข้อง": "✔" if r >= like_thr else "",
                       **({"เนื้อหา": item_content(i)} if item_content else {})}
                      for i, r in zip(te["i"], te["r"])])
    top = mark_hits(side_by_side_topk(recs_by_model, u, item_title, K), rel_titles)
    return {"history": h, "ground_truth": g, "recommended": top}


def build_recommender_form(user_ids, model_names, recommend_fn, user_info_fn, K_default=5, K_max=20,
                           title="ตัวอย่างการประยุกต์ใช้: ฟอร์มแนะนำภาพยนตร์"):
    """ฟอร์ม GUI ด้วย ipywidgets (มีใน Colab อยู่แล้ว)
    recommend_fn(user_id, model_name, K) → DataFrame รายการที่แนะนำ (พร้อมเนื้อหา/คะแนน)
    user_info_fn(user_id)                → DataFrame ข้อมูล/ประวัติของผู้ใช้
    """
    try:
        import ipywidgets as W
        from IPython.display import display, HTML, clear_output
    except Exception:
        print("ไม่มี ipywidgets — ติดตั้งด้วย pip install ipywidgets")
        return None
    w_user = W.Dropdown(options=list(user_ids), description="ผู้ใช้", layout=W.Layout(width="260px"))
    w_model = W.Dropdown(options=list(model_names), description="โมเดล", layout=W.Layout(width="320px"))
    w_k = W.IntSlider(value=K_default, min=1, max=K_max, description="K", continuous_update=False)
    btn = W.Button(description="แนะนำ", button_style="success", icon="play")
    out = W.Output()

    def run(_=None):
        with out:
            clear_output()
            display(HTML(f"<h4 style='color:#2E6B4A'>ข้อมูลผู้ใช้ {w_user.value}</h4>"))
            display(user_info_fn(w_user.value))
            display(HTML(f"<h4 style='color:#2E6B4A'>รายการที่แนะนำ Top-{w_k.value} โดย {w_model.value}</h4>"))
            display(recommend_fn(w_user.value, w_model.value, w_k.value))
    btn.on_click(run)
    w_user.observe(lambda c: run() if c["name"] == "value" else None, names="value")
    w_model.observe(lambda c: run() if c["name"] == "value" else None, names="value")
    box = W.VBox([W.HTML(f"<h3 style='color:#2E6B4A'>{title}</h3>"), W.HBox([w_user, w_model, w_k, btn]), out])
    display(box)
    run()
    return box

# ===================== cbrec_core (ฝัง) — จบ =====================

In [ ]:
set_seed(SEED)
print("การตั้งค่า:", dict(FAST_MODE=FAST_MODE, SEED=SEED, K=K, LIKE_THR=LIKE_THR, K_CORE=K_CORE, USE_SBERT=USE_SBERT))

## Part 1 · โหลด MovieLens และสร้าง "เนื้อหา" ของรายการ
**ข้อสังเกต** — MovieLens ไม่มีเรื่องย่อ เนื้อหาที่ใช้ได้คือ *ชื่อเรื่อง + แนว + tag ที่ผู้ใช้ติด* (มี tag เพียง ~1,500 เรื่อง)
นี่คือข้อจำกัดข้อที่ 2 ในตาราง 3.66 ("ความครบถ้วนของเขตข้อมูลเนื้อหา") ซึ่งจะสะท้อนในผลทุกตัว

In [ ]:
def load_movielens(data_dir=DATA_DIR):
    """ดาวน์โหลด ml-latest-small: ลอง GroupLens (zip) → สำรอง GitHub mirror (raw csv) → ใช้ไฟล์ในเครื่องถ้ามี"""
    files = ["ratings.csv", "movies.csv", "tags.csv", "links.csv"]
    os.makedirs(data_dir, exist_ok=True)
    if not all(os.path.exists(os.path.join(data_dir, f)) for f in files):
        try:
            url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
            print("กำลังดาวน์โหลดจาก GroupLens …")
            z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=60).read()))
            for f in files:
                with open(os.path.join(data_dir, f), "wb") as fh:
                    fh.write(z.read(f"ml-latest-small/{f}"))
        except Exception as e:
            print("GroupLens ไม่สำเร็จ:", str(e)[:80], "→ ใช้ mirror บน GitHub")
            base = "https://raw.githubusercontent.com/smanihwr/ml-latest-small/master/"
            for f in files:
                urllib.request.urlretrieve(base + f, os.path.join(data_dir, f))
    r = pd.read_csv(os.path.join(data_dir, "ratings.csv"))
    m = pd.read_csv(os.path.join(data_dir, "movies.csv"))
    t = pd.read_csv(os.path.join(data_dir, "tags.csv"))
    return r, m, t

ratings_raw, movies, tags = load_movielens()
print(f"ratings {ratings_raw.shape} · movies {movies.shape} · tags {tags.shape}")
print("ช่วงเวลา:", pd.to_datetime(ratings_raw.timestamp.min(), unit="s").date(), "→", pd.to_datetime(ratings_raw.timestamp.max(), unit="s").date())
display(ratings_raw.head(3)); display(movies.head(3)); display(tags.head(3))

In [ ]:
# ---------- 1.2 สร้างเขตข้อมูลเนื้อหาของรายการ: title_clean · year · genre tokens · tags ----------
import re as _re
def split_title(t):
    m = _re.match(r"^(.*?)\s*\((\d{4})\)\s*$", t)
    title, year = (m.group(1), int(m.group(2))) if m else (t, np.nan)
    title = _re.sub(r"^(.*), (The|A|An)$", r"\2 \1", title.strip())      # "Matrix, The" → "The Matrix"
    return title, year
movies[["title_clean", "year"]] = movies["title"].apply(lambda t: pd.Series(split_title(t)))
# แนว: "Sci-Fi" → scifi, "Film-Noir" → filmnoir (ตัวตัดคำ [a-z]+ จะไม่แยกคำ) ; "(no genres listed)" → ว่าง
def genre_tokens(g):
    return " ".join(x.replace("-", "").replace(" ", "").lower() for x in g.split("|") if not x.startswith("("))
movies["genre_tok"] = movies["genres"].apply(genre_tokens)
# tag: รวมทุก tag ของเรื่องนั้น (ตัวพิมพ์เล็ก) — เนื้อหาที่ "ผู้ใช้เขียน" จึงหลากหลายและมีสัญญาณรบกวน
tag_text = tags.groupby("movieId")["tag"].apply(lambda s: " ".join(str(x).lower() for x in s)).rename("tag_text")
movies = movies.merge(tag_text, on="movieId", how="left").fillna({"tag_text": ""})
# เนื้อหารวม: แนวถูกใส่ซ้ำ 2 ครั้งเพื่อเพิ่มน้ำหนัก (ทางเลือกอย่างง่ายของ BM25F/field weight — Lab 3-3 จะปรับจูนค่านี้)
movies["text"] = (movies["title_clean"] + " " + movies["genre_tok"] + " " + movies["genre_tok"] + " " + movies["tag_text"]).str.strip()
print(f"เรื่องที่มี tag: {(movies.tag_text != '').sum()} / {len(movies)} ({(movies.tag_text != '').mean():.1%})")
display(movies[["movieId", "title_clean", "year", "genres", "tag_text", "text"]].sample(5, random_state=SEED))

In [ ]:
# ---------- 1.3 ตรวจสอบข้อมูล (audit) ก่อนทำอะไรทั้งสิ้น: การกระจายคะแนน · ความนิยม (long tail) · ประวัติต่อผู้ใช้ ----------
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ratings_raw["rating"].value_counts().sort_index().plot.bar(ax=ax[0], color="#2E6B4A"); ax[0].set_title("Rating distribution"); ax[0].set_xlabel("rating")
ic = ratings_raw.groupby("movieId").size().sort_values(ascending=False).values
ax[1].plot(np.arange(1, len(ic) + 1), ic, color="#2E6B4A"); ax[1].set_yscale("log"); ax[1].set_xscale("log"); ax[1].set_title("Item popularity (long tail)"); ax[1].set_xlabel("item rank"); ax[1].set_ylabel("#ratings")
uc = ratings_raw.groupby("userId").size()
ax[2].hist(uc, bins=50, color="#2E6B4A"); ax[2].set_yscale("log"); ax[2].set_title("Ratings per user"); ax[2].set_xlabel("#ratings")
plt.tight_layout(); plt.show()
cum = np.cumsum(ic) / ic.sum()
print(f"ผู้ใช้ {ratings_raw.userId.nunique()} คน · รายการที่มีคะแนน {ratings_raw.movieId.nunique()} · ความหนาแน่น {len(ratings_raw)/(ratings_raw.userId.nunique()*len(movies)):.2%}")
print(f"รายการ head ที่สะสม 1/3 ของปฏิสัมพันธ์: {int((cum < 1/3).sum())+1} เรื่อง (~{((cum < 1/3).sum()+1)/len(ic):.1%} ของรายการที่มีคะแนน) · ค่ามัธยฐานประวัติ/คน = {uc.median():.0f}")
print(f"คะแนน ≥ 4 คิดเป็น {(ratings_raw.rating >= LIKE_THR).mean():.1%} · ≤ 2 คิดเป็น {(ratings_raw.rating <= DISLIKE_THR).mean():.1%}  ← ตัวอย่างลบชัดเจนมีน้อย")
# ข้อสังเกต: ความนิยมกระจุกมาก (หัวสั้น หางยาว) → TopPopular จะได้ Coverage ต่ำ/Gini สูงแน่นอน และ Novelty ต่ำ

## Part 2 · การแทนเนื้อหาของรายการ (หัวข้อ 3.4–3.7, 3.11, 3.13)

In [ ]:
# ---------- 2.1 TF-IDF (สูตรตั้งต้นของตำรา) บนคลังทั้ง 9,742 เรื่อง — min_df=2 ตัดคำที่พบเรื่องเดียว (ส่วนใหญ่คือ tag แปลก ๆ) ----------
with Timer("TF-IDF"):
    vec = TextVectorizer(tf="log", idf="smooth", min_df=2, max_df=0.85, norm="l2")
    X = vec.fit_transform(list(movies["text"]))
N_ITEMS = X.shape[0]
print(f"เมทริกซ์รายการ X: {X.shape} · ความหนาแน่น {X.nnz/(X.shape[0]*X.shape[1]):.3%} · avgdl {vec.avgdl_:.2f} โทเคน/เรื่อง")
zero_rows = int((np.asarray(X.sum(axis=1)).ravel() == 0).sum())
print(f"เรื่องที่เป็นเวกเตอร์ศูนย์ (ไม่มีคำในคลังศัพท์): {zero_rows} ← จะได้คะแนน 0 ทุกโปรไฟล์ (ตาราง 3.68 แถว NaN/ศูนย์)")
# คำที่ df สูงสุด 15 อันดับ — ตรวจว่า "คำเด่นไม่ใช่คำไวยากรณ์" (อาการข้อแรกในตาราง 3.68)
print("df สูงสุด:", sorted(vec.df_.items(), key=lambda x: -x[1])[:15])

# ---------- 2.2 เมทริกซ์แนว (0/1) — ปริภูมิ 19 มิติ สำหรับโปรไฟล์แนว, ILD และ Calibration ----------
GENRES = sorted({g for gs in movies["genres"] for g in gs.split("|") if not g.startswith("(")})
G = np.array([[1.0 if g in gs.split("|") else 0.0 for g in GENRES] for gs in movies["genres"]])
Gn = l2_normalize_rows(G)
print(f"แนว {len(GENRES)} มิติ:", GENRES)

In [ ]:
# ---------- 2.3 BM25 บนโทเคนเดียวกัน และ LSA k=100 บน TF-IDF (· Sentence-Embedding ถ้าเปิด USE_SBERT) ----------
tokens = vec._train_tokens
with Timer("BM25 fit"):
    bm25 = BM25(k1=1.5, b=0.75).fit(tokens)
with Timer("LSA k=100"):
    lsa = LSA(k=100, seed=SEED).fit(X); Z = lsa.transform(X)
print(f"LSA อธิบายความแปรปรวนได้ {lsa.explained_:.1%} ด้วย 100 มิติ")
E = sentence_embeddings(list(movies["text"])) if USE_SBERT else None
SEM, SEM_NAME = (E, "SBERT") if E is not None else (Z, "LSA")
# ตัวอย่าง: เพื่อนบ้านใกล้สุดของ "Toy Story" ในสามปริภูมิ — อ่านด้วยตาว่าปริภูมิใด "เข้าใจ" เนื้อหามากกว่า
def neighbors(M, idx, k=5):
    s = to_dense_row(l2_normalize_rows(M)[idx] @ l2_normalize_rows(M).T) if not sp.issparse(M) else to_dense_row(M @ M[idx].T)
    order = [i for i in np.argsort(-s) if i != idx][:k]
    return [f"{movies.title_clean[i]} ({s[i]:.2f})" for i in order]
q = int(movies.index[movies.title_clean == "Toy Story"][0])
show_df(pd.DataFrame({"TF-IDF": neighbors(X, q), "Genre": neighbors(G, q), SEM_NAME: neighbors(SEM, q)}), "เพื่อนบ้านของ Toy Story ในสามปริภูมิ", fmt="{}")
# ข้อสังเกต: ปริภูมิแนวให้ 'เสมอ' จำนวนมาก (เรื่องที่แนวเหมือนกันเป๊ะได้ 1.00 ทั้งหมด) → ต้องมีตัวตัดสินรอง (3.10.2)

## Part 3 · การเลือกกลุ่มตัวอย่างและการแบ่งข้อมูลสำหรับ Offline Training / Testing (หัวข้อ 2.7–2.9)

In [ ]:
# ---------- 3.1 k-core "ก่อน" แบ่งข้อมูล (ตาราง 2.10 แถว 3) และ (ทางเลือก) สุ่มผู้ใช้เมื่อ FAST_MODE ----------
# เทคนิคการเลือกกลุ่มตัวอย่างที่ใช้ในเล่มนี้ (สรุป):
#   ① k-core: ตัดผู้ใช้/รายการที่มีปฏิสัมพันธ์น้อยกว่า k ก่อนแบ่ง — ลดสัญญาณรบกวน แต่ทำให้งานง่ายขึ้น (ต้องประกาศ k)
#   ② สุ่ม "ผู้ใช้" ไม่สุ่ม "แถว" เมื่อต้องการชุดย่อย — ประวัติแต่ละคนยังครบ (FAST_MODE)
#   ③ แบ่ง train/valid/test ตามเวลาภายในผู้ใช้ (3.3) — valid ใช้เลือกโมเดล/hyperparameter, test แตะครั้งเดียว
#   ④ candidate pool = รายการที่อยู่ในชุดฝึก (คลังที่ให้บริการได้) เทียบกับทั้งคลัง และ sampled negatives (5.3)
#   ⑤ negative sampling สำหรับ "ฝึก" ตัวจำแนก: explicit (≤2) + implicit (สุ่มรายการที่ไม่เคยเห็น) (Part 4, LogReg)
#   ⑥ seed คงที่ทุกการสุ่ม และรายงาน seed variance (5.3)
df_all, u_map, i_map = encode_ids(ratings_raw, "userId", "movieId", items_df=movies, item_id_col="movieId")
df_all = df_all[["userId", "movieId", "u", "i", "r", "timestamp"]]
df_core = kcore_filter(df_all, k_user=K_CORE, k_item=K_CORE)
print(f"ก่อน k-core: {len(df_all)} แถว · ผู้ใช้ {df_all.u.nunique()} · รายการ {df_all.i.nunique()}")
print(f"หลัง {K_CORE}-core: {len(df_core)} แถว · ผู้ใช้ {df_core.u.nunique()} · รายการ {df_core.i.nunique()}  (ตัดรายการที่มีคะแนน < {K_CORE} ออก {df_all.i.nunique()-df_core.i.nunique()} เรื่อง)")
if FAST_MODE:
    rng = np.random.default_rng(SEED)
    keep_users = rng.choice(df_core.u.unique(), size=200, replace=False)    # สุ่มผู้ใช้ ไม่ใช่สุ่มแถว → ประวัติแต่ละคนยังครบ
    df_core = df_core[df_core.u.isin(keep_users)]
    print("FAST_MODE: เหลือผู้ใช้", df_core.u.nunique(), "คน")
# ข้อสังเกต: k-core ทำให้ "งานง่ายขึ้น" (ตัดรายการหายากที่ทำนายยากที่สุดออก) → ต้องประกาศค่า k ทุกครั้ง
#            คลังสำหรับ Coverage ยังคงเป็น N_ITEMS ทั้งหมด (9,742) เพื่อให้ตัวหารไม่เปลี่ยนตามการกรอง

In [ ]:
# ---------- 3.2 เปรียบเทียบโปรโตคอลการแบ่งข้อมูล 4 แบบ (2.8.1) บนข้อมูลเดียวกัน ----------
rows = []
splits = {"random": split_random(df_core, 0.2, seed=SEED),
          "global_temporal": split_global_temporal(df_core, 0.2),
          "per_user_temporal": split_per_user_temporal(df_core, 0.2),
          "leave_one_out": split_leave_one_out(df_core, "timestamp", only_relevant=True, like_thr=LIKE_THR, seed=SEED)}
for name, (tr, te) in splits.items():
    lk = leakage_check(tr, te)
    te_users = set(te.u); tr_users = set(tr.u)
    rows.append({"โปรโตคอล": name, "ฝึก": len(tr), "ทดสอบ": len(te), "ผู้ใช้ในชุดทดสอบ": len(te_users),
                 "ผู้ใช้ทดสอบที่ไม่มีประวัติฝึก (cold)": len(te_users - tr_users),
                 "เหตุการณ์อนาคตในชุดฝึก": lk["future_events_in_train"], "สัดส่วน": f"{lk['share']:.1%}",
                 "ผู้ใช้ที่มี rel≥4 ในชุดทดสอบ": sum(1 for _, s in relevant_from(te, LIKE_THR)[0].items() if s)})
show_df(pd.DataFrame(rows), "ตารางเปรียบเทียบโปรโตคอลการแบ่งข้อมูล — สังเกตคอลัมน์ cold และเหตุการณ์อนาคต", fmt="{}")
# ข้อสังเกต (สำคัญ):
#  * random: ชุดฝึกมีเหตุการณ์ 'หลัง' ชุดทดสอบเกือบทั้งหมด = ใช้อนาคตทำนายอดีต (รั่วไหลแบบที่ 1)
#  * global_temporal: ปลอดภัยที่สุดเชิงเวลา แต่ใน MovieLens ผู้ใช้ส่วนใหญ่ให้คะแนนใน 'ช่วงสั้น ๆ' → ผู้ใช้ทดสอบเกือบทั้งหมด
#    ไม่มีประวัติฝึกเลย (cold user) ซึ่งระบบอิงเนื้อหาสร้างโปรไฟล์ไม่ได้ → ประเมินได้แต่ TopPopular
#  * per_user_temporal: ทุกคนมีประวัติ + ทดสอบด้วยอนาคต 'ของตนเอง' — ยังรั่วข้ามผู้ใช้ (ไม่กระทบโมเดลอิงเนื้อหาล้วน
#    แต่กระทบสถิติความนิยม) → เป็นโปรโตคอลหลักของเล่มนี้ และประกาศข้อจำกัดนี้ไว้
#  * leave_one_out: ทุกคนมีเป้าหมาย 1 ชิ้น ความแปรปรวนสูง — ใช้เป็นการทดลองเสริม

In [ ]:
# ---------- 3.3 โปรโตคอลหลัก: per-user temporal สามชุด (train / valid / test) ----------
# test = 20% ท้ายของแต่ละคน · valid = ~12.5% ท้ายของที่เหลือ (=10% ของทั้งหมด) · train = ส่วนแรก
train, valid, test = three_way_split(df_core, protocol="per_user_temporal", valid_frac=0.1, test_frac=0.2)
for nm, d in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{nm:5s}: {len(d):6d} แถว · ผู้ใช้ {d.u.nunique()} · เวลา {pd.to_datetime(d.timestamp.min(), unit='s').date()} → {pd.to_datetime(d.timestamp.max(), unit='s').date()}")
# ตรวจว่าไม่มีคู่ (u,i) ซ้ำข้ามชุด และลำดับเวลาภายในผู้ใช้ถูกต้อง
assert len(set(zip(train.u, train.i)) & set(zip(test.u, test.i))) == 0
per_user_ok = all(train[train.u == u].timestamp.max() <= test[test.u == u].timestamp.min() for u in test.u.unique()[:50])
print("ลำดับเวลาภายในผู้ใช้ (ฝึกก่อนทดสอบ) ถูกต้อง:", per_user_ok)
N_USERS = int(df_core.u.max()) + 1
# โครงสร้างที่ทุกโมเดล/ตัวประเมินใช้ — สร้างจาก train เท่านั้น
train_hist = hist_from(train)
valid_rel, valid_grades = relevant_from(valid, LIKE_THR)
users_valid = sorted(u for u, s in valid_rel.items() if s and u in train_hist)      # ผู้ใช้ที่ประเมินได้จริง
print(f"ผู้ใช้ที่มีรายการเกี่ยวข้อง (≥{LIKE_THR}) ในชุดตรวจสอบและมีประวัติฝึก: {len(users_valid)} คน")
# สถิติจากชุดฝึกเท่านั้น (2.8.4): ความนิยม p(i), head/tail
pop = ItemPopularity(train, N_ITEMS, N_USERS)
head = head_items_from_train(pop.count, N_ITEMS, share=1 / 3)
print(f"head items (สะสม 1/3 ของปฏิสัมพันธ์ในชุดฝึก) = {len(head)} เรื่อง เช่น", [movies.title_clean[i] for i in list(head)[:5]])
# Candidate pool = "คลังที่ให้บริการได้" = รายการที่ปรากฏในชุดฝึก (k-core) — ประกาศในขั้นที่ 7 ; เทียบกับทั้งคลังใน Part 5.3
POOL = set(int(i) for i in train.i.unique())
print(f"candidate pool (รายการในชุดฝึก): {len(POOL)} เรื่อง จากทั้งคลัง {N_ITEMS}")
# กลุ่มผู้ใช้สำหรับ fairness: ประวัติยาว/สั้น ตามมัธยฐานของชุดฝึก
hist_len = train.groupby("u").size()
GROUP = {u: ("long" if hist_len.get(u, 0) >= hist_len.median() else "short") for u in range(N_USERS)}

In [ ]:
# ---------- 3.4 Protocol Card — ประกาศก่อนดูผล ----------
card = ProtocolCard({
    "1 เป้าหมายผู้ใช้/ธุรกิจ": "ผู้ใช้พบเรื่องที่จะให้ ≥4 ดาวใน 10 อันดับแรก · ธุรกิจ: เพิ่มการค้นพบเรื่องนอกกระแส (Coverage/LongTail) โดยไม่ลด nDCG",
    "2 Primary Metric": f"nDCG@{K} (gain = linear ตาม rating)",
    "3 Guardrail Metrics": f"Coverage@{K} ต้องไม่ต่ำกว่า TFIDF-centroid เกิน 10% · Gini ห้ามเพิ่มเกิน 0.02 · ILD@{K} รายงานเสมอ",
    "4 สูตรและค่า K / นิยามความเกี่ยวข้อง": f"K={K} · relevant = rating ≥ {LIKE_THR} · TF=1+log2 f · IDF=ln((1+N)/(1+df))+1 · L2 · min_df=2 · max_df=0.85",
    "5 การแบ่งข้อมูล": f"{K_CORE}-core → per-user temporal: train {len(train)} · valid {len(valid)} · test {len(test)} (เลือกโมเดลบน valid, แตะ test ครั้งเดียว)",
    "6 การป้องกันรั่วไหล": "ความนิยม / head-tail / p(i) / โปรไฟล์ ทุกตัวจาก train · IDF จากคลังรายการ (ไม่ใช่ข้อมูลผู้ใช้) · ยอมรับการรั่วข้ามผู้ใช้ของ per-user split และประกาศ",
    "7 Candidate Set": f"รายการที่อยู่ในชุดฝึกหลัง {K_CORE}-core ({len(POOL)} เรื่อง) หักที่เห็นใน train · ตัวตัดสินรองเมื่อคะแนนเสมอ = ความนิยม (3.10.2) · เทียบกับทั้งคลัง 9,742 และ sampled m=99 ใน Part 5.3",
    "8 Negative Sampling": "ประเมิน: ไม่ใช้ (full metric) · ฝึก LR: explicit (≤2) + สุ่มรายการที่ไม่เคยเห็น 1:1",
    "9 Accuracy + Beyond-Accuracy": "รายงานทั้งสองด้านทุกโมเดล (Part 5–6)",
    "10 ความไม่แน่นอน": "paired bootstrap ระดับผู้ใช้ 2,000 รอบ · Holm–Bonferroni · seed 3 ค่า (sampled candidates)",
    "11 ตรวจกับผู้ใช้จริง": "ออฟไลน์ = คัดกรอง ; ต้อง A/B test ก่อนสรุป"})
card.show()

## Part 4 · โมเดลทั้งหมด (score function ต่อผู้ใช้) — บทที่ 3
ทุกโมเดลรับ `u` แล้วคืนคะแนนของ **ทุกรายการ** (ยาว 9,742) โดยใช้เฉพาะ `train_hist[u]` · ผู้ใช้ที่ไม่มีประวัติจะได้ None → คะแนนศูนย์
(ตัวประเมินจัดอันดับ กรองที่เคยเห็น และตัดสินเสมอด้วยดัชนีรายการให้เอง)

In [ ]:
S_item = None   # เมทริกซ์ความคล้ายเต็ม 9.7k×9.7k (dense ~760 MB) ไม่สร้าง — Item-kNN คำนวณเฉพาะแถวที่ต้องใช้
C_counts = TextVectorizer(tf="raw", idf="none", norm=None, min_df=2, max_df=0.85).fit_transform(list(movies["text"]))   # สำหรับ NB
rng_train = np.random.default_rng(SEED)

def make_models(train_hist, pop, seed=SEED):
    """คืน dict ชื่อโมเดล → score_fn(u) ผูกกับ train_hist ที่ให้มา (เรียกใหม่เมื่อฝึกด้วย train+valid ใน Part 8)"""
    rng = np.random.default_rng(seed)
    cache = {}
    pop_tb = 1e-6 * minmax(pop.p)          # ตัวตัดสินรอง (secondary key): เสมอกันให้เรื่องที่นิยมกว่าขึ้นก่อน (3.10.2) — เล็กมากจนไม่เปลี่ยนลำดับที่ไม่เสมอ
    def tb(s):
        return None if s is None else np.asarray(s, dtype=np.float64) + pop_tb
    def profile(u, method="centroid", M=X, **kw):
        key = (u, method, id(M), tuple(sorted(kw.items())))
        if key not in cache:
            cache[key] = build_profile(M, train_hist.get(u, {}), method, LIKE_THR, DISLIKE_THR, **kw)
        return cache[key]
    def m_random(u):        return rng.random(N_ITEMS)
    def m_pop(u):           return pop.p.copy()
    def m_genre(u):         return tb(match_scores(Gn, profile(u, "centroid", G)))
    def m_tfidf(u):         return tb(match_scores(X, profile(u, "centroid", X)))
    def m_weighted(u):      return tb(match_scores(X, profile(u, "weighted", X)))
    def m_rocchio(u):       return tb(match_scores(X, profile(u, "rocchio", X, beta=0.8, gamma=0.2)))
    def m_sem(u):           return tb(match_scores(SEM, profile(u, "centroid", SEM)))
    def m_bm25(u):
        p = profile(u, "centroid", X)
        top = vec.top_terms(p, 20)                                   # ข้อคำถาม = คำเด่น 20 คำ ถ่วงด้วยน้ำหนักโปรไฟล์
        return tb(bm25.score_query([t for t, _ in top], dict(top))) if top else None
    def m_knn(u):
        liked = [i for i, r in train_hist.get(u, {}).items() if r >= LIKE_THR]
        if not liked: return None
        Sl = np.asarray((X @ X[liked].T).todense())                  # n_items × |liked| เฉพาะคอลัมน์ที่ต้องใช้
        return tb(item_knn_scores(Sl, list(range(len(liked))), "mean", k=10))
    def m_nb(u):
        s = ClassifierProfile("nb", alpha=0.5, like_thr=LIKE_THR, dislike_thr=DISLIKE_THR, min_examples=3).fit(C_counts, train_hist.get(u, {})).score(C_counts)
        return tb(s) if s is not None else m_tfidf(u)                # fallback บันได 3.48: ไม่มีตัวอย่างลบ → centroid
    def m_lr(u):
        h = train_hist.get(u, {})
        pos = [i for i, r in h.items() if r >= LIKE_THR]
        if len(pos) < 3: return m_tfidf(u)
        # negative sampling สำหรับ "ฝึก": ตัวอย่างลบชัดเจน (≤2) + รายการที่ไม่เคยเห็นสุ่มเท่าจำนวนบวก (implicit negatives)
        neg = [i for i, r in h.items() if r <= DISLIKE_THR]
        unseen = np.setdiff1d(np.arange(N_ITEMS), list(h))
        neg = neg + list(rng.choice(unseen, size=len(pos), replace=False))
        from sklearn.linear_model import LogisticRegression
        clf = LogisticRegression(C=1.0, max_iter=500).fit(X[pos + neg], [1] * len(pos) + [0] * len(neg))
        return tb(clf.predict_proba(X)[:, 1])
    def m_hybrid(u):        return tb(rrf_fusion([m_tfidf(u), m_genre(u), m_sem(u)], k=60))
    def m_hybrid_pop(u):    return convex_fusion([m_hybrid(u), pop.p], weights=[0.8, 0.2])     # ผสมความนิยมเป็น "ตัวตัดสินรอง"
    return {"Random": m_random, "TopPopular": m_pop, "Genre-centroid": m_genre, "TFIDF-centroid": m_tfidf,
            "TFIDF-weighted": m_weighted, "TFIDF-Rocchio": m_rocchio, "BM25-profile": m_bm25, f"{SEM_NAME}-centroid": m_sem,
            "ItemKNN-TFIDF": m_knn, "NaiveBayes": m_nb, "LogReg(+neg sampling)": m_lr,
            "Hybrid-RRF(tfidf+genre+sem)": m_hybrid, "Hybrid+Pop(0.2)": m_hybrid_pop}

MODELS = make_models(train_hist, pop)
# MMR บน Hybrid: ความคล้ายเพื่อความหลากหลาย = ปริภูมิแนว (คำนวณเฉพาะ pool ของผู้ใช้นั้น)
def make_mmr(score_fn, lam=0.7, feat=Gn, pool=None):
    def rerank(u, s):
        seen = set(train_hist.get(u, {}))
        base = top_k(s, seen, K=5 * K, candidates=pool)
        if not base: return []
        Sm = feat[base] @ feat[base].T
        sub = mmr_rerank(s[base], np.asarray(Sm), K=K, lam=lam, pool=len(base))
        return [base[j] for j in sub]
    return rerank
print("โมเดลที่จะเปรียบเทียบ:", list(MODELS) + ["Hybrid+MMR(λ=0.7)"])
# ทดสอบเวลาต่อผู้ใช้หนึ่งคนของแต่ละโมเดล
u0 = users_valid[0]
for nm, fn in MODELS.items():
    with Timer(nm):
        fn(u0)

## Part 5 · ประเมินแบบ Ranking บนชุดตรวจสอบ (Validation) — Accuracy และ Beyond-Accuracy

In [ ]:
# ---------- 5.1 รันทุกโมเดลด้วยตัวประเมินเดียวกัน (full catalog, K=10) ----------
def run_all(models, users, hist, rel, grades, extra_rerank=None, candidate="full", seed=SEED, label="", pool=None):
    summaries, per_users, recs_all = {}, {}, {}
    for name, fn in models.items():
        with Timer(f"{label}{name}"):
            pu, sm, recs = evaluate_ranking(fn, users, hist, rel, N_ITEMS, K=K, item_feat=G, pop=pop, head=head, genre_mat=G,
                                            rel_grades=grades, candidate=candidate, m_neg=99, seed=seed, pool=pool)
        summaries[name], per_users[name], recs_all[name] = sm, pu, recs
    for name, (fn, rr) in (extra_rerank or {}).items():
        with Timer(f"{label}{name}"):
            pu, sm, recs = evaluate_ranking(fn, users, hist, rel, N_ITEMS, K=K, item_feat=G, pop=pop, head=head, genre_mat=G,
                                            rel_grades=grades, rerank_fn=rr, seed=seed, pool=pool)
        summaries[name], per_users[name], recs_all[name] = sm, pu, recs
    return summaries, per_users, recs_all

extra = {"Hybrid+MMR(λ=0.7)": (MODELS["Hybrid-RRF(tfidf+genre+sem)"], make_mmr(MODELS["Hybrid-RRF(tfidf+genre+sem)"], 0.7, pool=POOL))}
summ_v, pu_v, recs_v = run_all(MODELS, users_valid, train_hist, valid_rel, valid_grades, extra_rerank=extra, label="valid · ", pool=POOL)
tbl_v = results_table(summ_v, K=K)
show_df(tbl_v, f"ผล Ranking บน Validation (K={K}, candidate pool {len(POOL)} เรื่อง, ผู้ใช้ {len(users_valid)} คน)")

In [ ]:
# ---------- 5.2 กราฟ: Accuracy · Beyond-Accuracy · การแลกเปลี่ยน ----------
plot_metric_bars(tbl_v, [f"P@{K}", f"R@{K}", f"HR@{K}", f"MRR@{K}", f"MAP@{K}", f"nDCG@{K}", "AUC"], title="Accuracy (validation)", ncols=4)
plot_metric_bars(tbl_v, [f"Coverage@{K}", "Gini", f"LongTail@{K}", f"ILD@{K}", f"Novelty@{K}", "CalibErr"], title="Beyond-accuracy (validation)", ncols=3)
plot_tradeoff(tbl_v, x=f"nDCG@{K}", y=f"Coverage@{K}", title=f"nDCG@{K} vs Coverage@{K}")
plot_tradeoff(tbl_v, x=f"nDCG@{K}", y=f"Novelty@{K}", title=f"nDCG@{K} vs Novelty@{K}")
# วิธีอ่าน: TopPopular มักชนะด้าน accuracy บน MovieLens (อคติความนิยมของข้อมูล) แต่ Coverage/Novelty/LongTail แย่ที่สุด
#           โมเดลอิงเนื้อหาให้ Coverage/Novelty สูงกว่าหลายเท่า — นี่คือเหตุผลที่ต้องวัด "ทั้งสองด้าน" (2.4.2)

In [ ]:
# ---------- 5.3 Candidate set สามแบบ (2.9): pool k-core (หลัก) · ทั้งคลัง 9,742 · sampled negatives m=99 — ตัวเลขและ "ลำดับผู้ชนะ" เปลี่ยนหรือไม่ ----------
SUB = ["TopPopular", "TFIDF-centroid", "TFIDF-Rocchio", "BM25-profile", f"{SEM_NAME}-centroid", "NaiveBayes", "Hybrid-RRF(tfidf+genre+sem)"]
summ_full, _, _ = run_all({k: MODELS[k] for k in SUB}, users_valid, train_hist, valid_rel, valid_grades, label="full · ", pool=None)
summ_s, _, _ = run_all({k: MODELS[k] for k in SUB}, users_valid, train_hist, valid_rel, valid_grades, candidate="sampled", label="sampled · ", pool=POOL)
cmp = pd.DataFrame({f"pool({len(POOL)}) nDCG@10": {k: summ_v[k]["nDCG@K"] for k in SUB}, "full(9742) nDCG@10": {k: summ_full[k]["nDCG@K"] for k in SUB},
                    "sampled(m=99) nDCG@10": {k: summ_s[k]["nDCG@K"] for k in SUB}, f"pool HR@10": {k: summ_v[k]["HR@K"] for k in SUB},
                    "full HR@10": {k: summ_full[k]["HR@K"] for k in SUB}, "sampled HR@10": {k: summ_s[k]["HR@K"] for k in SUB}})
for c in [f"pool({len(POOL)}) nDCG@10", "full(9742) nDCG@10", "sampled(m=99) nDCG@10"]:
    cmp["อันดับ " + c.split(" ")[0]] = cmp[c].rank(ascending=False).astype(int)
show_df(cmp, "ขนาดของ candidate set เปลี่ยนทั้งตัวเลขและลำดับผู้ชนะ (ตัวอย่าง 2.14–2.15) — ต้องประกาศทุกครั้ง")
# seed variance ของ sampled metric (2.10.1): เปลี่ยนเฉพาะ seed ของการสุ่มรายการลบ
sv, mu_, sd_ = seed_variance(lambda s: evaluate_ranking(MODELS["TFIDF-centroid"], users_valid, train_hist, valid_rel, N_ITEMS, K=K,
                                                        rel_grades=valid_grades, candidate="sampled", m_neg=99, seed=s, pool=POOL)[1], seeds=[1, 2, 3])
show_df(sv[["seed", "nDCG@K", "HR@K"]], f"sampled nDCG@10 ของ TFIDF-centroid ตาม seed: {mu_:.4f} ± {sd_:.4f}")
# ข้อสังเกต: ห้ามนำ sampled metric ไปเทียบกับ full metric ของงานอื่น และต้องระบุ m + วิธีสุ่ม (2.9.5)

In [ ]:
# ---------- 5.4 ความไม่แน่นอน: paired bootstrap ระดับผู้ใช้ + Holm–Bonferroni (2.10) ----------
BASE = "TFIDF-centroid"
rows, pvals, names = [], [], []
for name in tbl_v.index:
    if name == BASE: continue
    a = pu_v[BASE].set_index("user")["nDCG@K"]; b = pu_v[name].set_index("user")["nDCG@K"].reindex(a.index)
    bs = paired_bootstrap(a.values, b.values, n_boot=2000, seed=SEED)
    rows.append({"โมเดล": name, f"Δ nDCG@{K} เทียบ {BASE}": bs["diff"], "CI 95% ล่าง": bs["ci_low"], "CI 95% บน": bs["ci_high"], "p": bs["p_value"]})
    pvals.append(bs["p_value"]); names.append(name)
rej = holm_bonferroni(pvals, alpha=0.05)
boot = pd.DataFrame(rows); boot["Holm: มีนัยสำคัญ"] = ["✔" if r else "" for r in rej]
show_df(boot.set_index("โมเดล"), f"paired bootstrap (ผู้ใช้ {len(users_valid)} คน, 2,000 รอบ) — Holm คุม family-wise error ของ {len(pvals)} การเปรียบเทียบ")
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(boot[f"Δ nDCG@{K} เทียบ {BASE}"], range(len(boot)), xerr=[boot[f"Δ nDCG@{K} เทียบ {BASE}"] - boot["CI 95% ล่าง"], boot["CI 95% บน"] - boot[f"Δ nDCG@{K} เทียบ {BASE}"]], fmt="o", color="#2E6B4A")
ax.axvline(0, color="grey", ls="--"); ax.set_yticks(range(len(boot))); ax.set_yticklabels(boot["โมเดล"], fontsize=8); ax.set_xlabel(f"nDCG@{K} difference vs {BASE} (95% CI)"); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ---------- 5.5 Fairness ฝั่งผู้ใช้ (2.5.6): Parity gap ของ nDCG ระหว่างผู้ใช้ประวัติยาว/สั้น ----------
rows = []
for name in tbl_v.index:
    pu = pu_v[name].copy(); pu["group"] = pu["user"].map(GROUP)
    gap, g = parity_gap(pu, "nDCG@K", "group")
    rows.append({"โมเดล": name, "nDCG@K long": g.get("long", np.nan), "nDCG@K short": g.get("short", np.nan), "Parity gap": gap})
show_df(pd.DataFrame(rows).set_index("โมเดล"), "nDCG@10 แยกกลุ่มประวัติยาว (≥ มัธยฐาน) / สั้น — ระบบอิงเนื้อหาช่วยผู้ใช้ประวัติสั้นได้ดีกว่าหรือไม่?")

## Part 6 · ประเมินแบบ Rating (MAE / RMSE) บนชุดตรวจสอบ (หัวข้อ 2.1)
โมเดลอิงเนื้อหาสำหรับ "ทำนายคะแนน" = Content-kNN regression: r̂ = μ_u + Σ sim·(r−μ_u)/Σ|sim| บนเพื่อนบ้าน k ตัวที่ผู้ใช้เคยให้คะแนน

In [ ]:
rating_models = {"GlobalMean": MeanRatingBaseline("global"), "UserMean": MeanRatingBaseline("user"), "User+ItemBias": MeanRatingBaseline("user_item"),
                 "ContentKNN-TFIDF(k=20)": ContentKNNRegressor(X, k=20), "ContentKNN-Genre(k=20)": ContentKNNRegressor(G, k=20),
                 f"ContentKNN-{SEM_NAME}(k=20)": ContentKNNRegressor(SEM, k=20), "ContentKNN-TFIDF(k=5)": ContentKNNRegressor(X, k=5)}
pred_v = {}
for name, m in rating_models.items():
    with Timer(name):
        pred_v[name] = np.clip(m.fit(train).predict(valid), 0.5, 5.0)
tbl_r = pd.DataFrame({n: {"MAE": mae(valid["r"], p), "RMSE": rmse(valid["r"], p)} for n, p in pred_v.items()}).T
show_df(tbl_r, f"MAE / RMSE บน Validation ({len(valid)} คู่) — ยิ่งน้อยยิ่งดี")
plot_metric_bars(tbl_r, ["MAE", "RMSE"], title="Rating prediction (validation)", ncols=2)
# ข้อสังเกต: ระบบเนื้อหาแบบ kNN มักชนะ UserMean เพียงเล็กน้อยบน MovieLens เพราะ 'เนื้อหา' (ชื่อ+แนว+tag) บอกรสนิยมได้จำกัด
#            และ k เล็กทำให้ RMSE แย่ลง (ความผันผวนจากเพื่อนบ้านน้อย) — เทียบ k=5 กับ k=20

In [ ]:
# ตัวอย่างค่าทำนายรายคู่ของผู้ใช้หนึ่งคน (เทียบทุกโมเดล) — ดูว่าโมเดลใด "กล้า" ต่างจากค่าเฉลี่ย และผิดทางไหน
u_show = users_valid[len(users_valid) // 2]
mask = (valid["u"] == u_show).values
ex = valid[mask][["i", "r"]].copy().rename(columns={"r": "จริง"})
ex.insert(0, "รายการ", [movies.title_clean[i] for i in ex["i"]]); ex.insert(1, "แนว", [movies.genres[i] for i in ex["i"]])
for n, p in pred_v.items():
    ex[n] = np.round(p[mask], 2)
show_df(ex.drop(columns="i").head(12), f"ค่าทำนายของผู้ใช้ {u_show} (12 คู่แรกในชุดตรวจสอบ)", fmt="{}")

## Part 7 · ตัวอย่างผลลัพธ์บนชุดตรวจสอบ: เปรียบเทียบ Top-10 ของแต่ละอัลกอริทึม
แสดงผู้ใช้ 3 คน: ประวัติยาว · ประวัติสั้น · สุ่ม — ✔ = อยู่ในของจริง (rating ≥ 4 ในชุดตรวจสอบ)

In [ ]:
def title_of(i):   return f"{movies.title_clean[i]} ({int(movies.year[i]) if pd.notna(movies.year[i]) else '—'})"
def content_of(i): return f"{movies.genres[i]}" + (f" · tags: {movies.tag_text[i][:50]}" if movies.tag_text[i] else "")
SHOW = ["TopPopular", "Genre-centroid", "TFIDF-centroid", "TFIDF-Rocchio", "BM25-profile", f"{SEM_NAME}-centroid", "LogReg(+neg sampling)", "Hybrid-RRF(tfidf+genre+sem)", "Hybrid+MMR(λ=0.7)"]
rng = np.random.default_rng(SEED)
u_long = max(users_valid, key=lambda u: hist_len.get(u, 0)); u_short = min(users_valid, key=lambda u: hist_len.get(u, 0)); u_rand = int(rng.choice(users_valid))
for label, u in [("ประวัติยาวที่สุด", u_long), ("ประวัติสั้นที่สุด", u_short), ("สุ่ม", u_rand)]:
    rep = user_report(u, train_hist, valid, {m: recs_v[m] for m in SHOW}, title_of, content_of, K=K, like_thr=LIKE_THR)
    print(f"\n{'='*30} ผู้ใช้ {u} ({label}) · ประวัติฝึก {len(train_hist[u])} เรื่อง · ของจริง ≥4 ในชุดตรวจสอบ {len(valid_rel[u])} เรื่อง {'='*30}")
    show_df(rep["history"].head(8), "① ประวัติ (ชุดฝึก, 8 เรื่องคะแนนสูงสุด)", fmt="{}")
    p = build_profile(X, train_hist[u], "centroid", LIKE_THR, DISLIKE_THR)
    print("   คำเด่นในโปรไฟล์ TF-IDF:", ", ".join(f"{t} {w}" for t, w in vec.top_terms(p, 10)))
    show_df(rep["ground_truth"].head(10), "② ของจริงในชุดตรวจสอบ (10 แถวแรก)", fmt="{}")
    show_df(rep["recommended"], "③ Top-10 ของแต่ละโมเดล (✔ = ตรงของจริง)", fmt="{}")

## Part 8 · ทดสอบ "ครั้งเดียว" บนชุดทดสอบ (ฝึกใหม่ด้วย train + valid)
เลือกโมเดล 4 ตัวจาก Validation (ตาม Primary + Guardrail) แล้วจึงแตะชุดทดสอบ — ห้ามย้อนกลับไปปรับอะไรอีกหลังเห็นผลนี้

In [ ]:
trainval = pd.concat([train, valid])
hist_tv = hist_from(trainval)
pop_tv = ItemPopularity(trainval, N_ITEMS, N_USERS)
head_tv = head_items_from_train(pop_tv.count, N_ITEMS)
POOL_TV = set(int(i) for i in trainval.i.unique())
test_rel, test_grades = relevant_from(test, LIKE_THR)
users_test = sorted(u for u, s in test_rel.items() if s and u in hist_tv)
MODELS_TV = make_models(hist_tv, pop_tv)
# เลือกจาก validation: ดีที่สุดตาม nDCG ในกลุ่ม personalized + ระบบฐาน TopPopular + TFIDF-centroid + MMR
best_personal = tbl_v.drop(index=["TopPopular", "Random"])[f"nDCG@{K}"].idxmax()
FINAL = list(dict.fromkeys(["TopPopular", "TFIDF-centroid", best_personal, "Hybrid-RRF(tfidf+genre+sem)"]))
print("โมเดลที่นำไปทดสอบ:", FINAL + ["Hybrid+MMR(λ=0.7)"])
extra_tv = {"Hybrid+MMR(λ=0.7)": (MODELS_TV["Hybrid-RRF(tfidf+genre+sem)"], make_mmr(MODELS_TV["Hybrid-RRF(tfidf+genre+sem)"], 0.7, pool=POOL_TV))}
pop, head, train_hist = pop_tv, head_tv, hist_tv      # ให้ตัวประเมิน/MMR ใช้สถิติจาก train+valid
summ_t, pu_t, recs_t = run_all({k: MODELS_TV[k] for k in FINAL}, users_test, hist_tv, test_rel, test_grades, extra_rerank=extra_tv, label="test · ", pool=POOL_TV)
tbl_t = results_table(summ_t, K=K)
show_df(tbl_t, f"ผลบนชุดทดสอบ (ผู้ใช้ {len(users_test)} คน) — รายงานครั้งเดียว")
a = pu_t["TopPopular"].set_index("user")["nDCG@K"]
for name in tbl_t.index:
    if name == "TopPopular": continue
    b = pu_t[name].set_index("user")["nDCG@K"].reindex(a.index)
    bs = paired_bootstrap(a.values, b.values, seed=SEED)
    print(f"{name:32s} Δ nDCG@{K} เทียบ TopPopular = {bs['diff']:+.4f}  CI95 [{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]  p={bs['p_value']:.3f}")
plot_metric_bars(tbl_t, [f"nDCG@{K}", f"HR@{K}", f"Coverage@{K}", f"Novelty@{K}", f"ILD@{K}", "Gini"], title="Test set (one-shot)", ncols=3)

## Part 9 · ฟอร์ม GUI สาธิตการประยุกต์ใช้งาน
เลือกผู้ใช้ (รหัส userId จริงของ MovieLens) · โมเดล · K → แสดง **ประวัติผู้ใช้พร้อมเนื้อหา** และ **รายการที่แนะนำ** พร้อมคะแนน แนว tag
และเหตุผล (คำร่วมกับโปรไฟล์) · ✔ = อยู่ในของจริงของชุดทดสอบ (โมเดลฝึกด้วย train+valid)

In [ ]:
inv_u = {v: k for k, v in u_map.items()}
GUI_MODELS = {**MODELS_TV}
GUI_USERS = [inv_u[u] for u in users_test]

def gui_user_info(user_id):
    u = u_map[user_id]
    h = hist_tv.get(u, {})
    rows = [{"รายการ": title_of(i), "คะแนน": r, "แนว": movies.genres[i], "tags": movies.tag_text[i][:40]} for i, r in sorted(h.items(), key=lambda x: -x[1])[:10]]
    p = build_profile(X, h, "centroid", LIKE_THR, DISLIKE_THR)
    print(f"ประวัติทั้งหมด {len(h)} เรื่อง (แสดง 10 เรื่องคะแนนสูงสุด) · กลุ่ม: {GROUP[u]} history")
    print("คำเด่นในโปรไฟล์:", ", ".join(f"{t} {w}" for t, w in vec.top_terms(p, 10)))
    print("ของจริงในชุดทดสอบ (≥4):", ", ".join(title_of(i) for i in list(test_rel.get(u, []))[:8]))
    return pd.DataFrame(rows)

def gui_recommend(user_id, model_name, Kk):
    u = u_map[user_id]
    if model_name == "Hybrid+MMR(λ=0.7)":
        s = GUI_MODELS["Hybrid-RRF(tfidf+genre+sem)"](u); rec = make_mmr(None, 0.7, pool=POOL_TV)(u, s)[:Kk]
    else:
        s = GUI_MODELS[model_name](u); s = np.zeros(N_ITEMS) if s is None else s
        rec = top_k(s, set(hist_tv.get(u, {})), Kk, candidates=POOL_TV)
    p = build_profile(X, hist_tv.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR)
    rows = []
    for rank, i in enumerate(rec, 1):
        why = ", ".join(t for t, *_ in explain_terms(vec, p, X[i], 3)) or "—"
        rows.append({"อันดับ": rank, "ตรงของจริง": "✔" if i in test_rel.get(u, set()) else "", "รายการ": title_of(i), "คะแนน": round(float(s[i]), 4),
                     "แนว": movies.genres[i], "ความนิยม p(i)": round(float(pop_tv[i]), 3), "คำร่วมกับโปรไฟล์": why, "tags": movies.tag_text[i][:40]})
    return pd.DataFrame(rows)

form = build_recommender_form(GUI_USERS, list(GUI_MODELS) + ["Hybrid+MMR(λ=0.7)"], gui_recommend, gui_user_info, K_default=K, K_max=30,
                              title="MovieLens · ระบบผู้แนะนำอิงเนื้อหา (สาธิตการประยุกต์ใช้)")

## Part 10 · สรุปข้อสังเกต

1. **โปรโตคอลกำหนดผล** — random split รั่วเวลาเกือบทั้งหมด · global temporal ทำให้ผู้ใช้ทดสอบเป็น cold user จนระบบอิงเนื้อหาไม่มีโปรไฟล์ ·
   per-user temporal คือทางสายกลางที่ต้องประกาศข้อจำกัด (Part 3.2)
2. **TopPopular แข็งแกร่งด้าน accuracy บน MovieLens** เพราะอคติความนิยมของข้อมูล แต่ Coverage/Novelty/LongTail ต่ำที่สุด —
   การรายงานเฉพาะ nDCG จะทำให้เลือกระบบที่ "แนะนำเรื่องเดิมให้ทุกคน"
3. **เนื้อหาบาง (ชื่อ+แนว+tag) จำกัดคุณภาพทุกโมเดล** — ตาราง 3.66 ข้อ 2 ; Lab 3-3 จะปรับน้ำหนักเขตข้อมูลและ hyperparameter
4. **Hybrid (RRF) มักได้สมดุลดีที่สุด** เพราะรวมหลักฐานจากสามปริภูมิโดยไม่ต้องปรับมาตรา ; MMR เพิ่ม ILD โดยแลก nDCG เล็กน้อย
5. **sampled candidates ทำให้ตัวเลขสูงขึ้นหลายเท่าและอาจสลับผู้ชนะ** — ต้องระบุ m และห้ามเทียบข้ามงาน
6. **CI จาก bootstrap + Holm** คือสิ่งที่ทำให้พูดได้ว่า "ต่างอย่างมีนัยสำคัญ" ไม่ใช่ทศนิยมตำแหน่งที่สาม
7. **Rating vs Ranking วัดคนละอย่าง** — โมเดลที่ RMSE ดีอาจจัดอันดับไม่ดี และกลับกัน ; เลือก Primary metric ตามเป้าหมายของระบบ